# Data center workbook vs PKG: can we find it in our own data?

Three questions, answered from the PKG tables in cdi_tm (read-only, no table is written):

- **A. Clients**: is every Power ID in the workbook a PKG customer, and does it have payment metrics and a credit row (DHE, PD)?
- **B. Counterparties**: do the names without a Power ID (XX / DCM / DCC ids) exist as counterparty entities with metrics?
- **C. Links and flows**: is each workbook link (data center ↔ second-order party) a direct payment relationship in PKG, and do the TDH amounts agree with ours?
- **Figures and regularity** (sections 10 to 12): figures of the findings saved as PNG files, and whether the payments on the found links are regular or casual.

How to run: keep the workbook next to this notebook, Run All. Results are saved under `dc_check_out/` so a rerun does not scan again (set `REUSE_SAVED = False` to force a fresh scan).
Matches by Power ID are **observed**; matches by company name are **inferred** and always shown separately.

In [ ]:
# ── 0. Setup ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")                      # ResourceWarning / DeprecationWarning / FutureWarning included
import re, html, time, json, hashlib, datetime, pickle
from pathlib import Path
import numpy as np
import pandas as pd

# compatibility shims (Cloudera PySpark 3.3 is older than the NumPy / pandas in new_default)
for _a, _t in (("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp), ("bool", bool)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import HTML, display
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
import openpyxl
from openpyxl.utils import get_column_letter

# ── config ──────────────────────────────────────────────────────────────────
XLSX_PATH   = "Datacenter Table Structure (v3).xlsx"
DB          = "bdahd01p_dlcdi1_cdi_tm"
T_CUST_DIM  = f"{DB}.pkg_cust_dim"                    # mdm_id ↔ customer_pwr_id / master_pwr_id
T_PAIR      = f"{DB}.pkg_cust_pair_month_v4"          # customer × dir × scope × counterparty × month
T_CUST_MET  = f"{DB}.pkg_cust_metrics_monthly_v4"     # customer payment metrics
T_CPTY_MET  = f"{DB}.pkg_cpty_metrics_monthly"        # counterparty entity metrics (ent_id = ent_l2)
T_TM        = f"{DB}.pkg_tm_cust_month"               # TM + credit block (DHE, PD) per power id × month
T_EDGE      = f"{DB}.pkg_edge_monthly"                # payer × payee × category × month (category number = rail)
T_MAP       = f"{DB}.pkg_cpty_entity_map_v2"          # counterparty id → entity (ent_l2)
T_EDGE_D    = f"{DB}.pkg_edge_daily"                  # payment days (timelines, regularity measured on the fly)
T_REG       = f"{DB}.pkg_cust_reg_stream"             # stored periodicity: one row per customer × direction × counterparty × month
T_CFACT     = "clhhd01p_clh.cdm_customer_fact_v"      # credit facts for Power IDs that pkg_tm_cust_month does not hold
T_PD        = "clhhd01p_clh.cdm_probability_default_dim_v"

PAIR_MONTHS = None        # None = every month of the pair table; or ("2025-01", "2026-10") to scan fewer months
NAME_MONTHS = 12          # counterparty names are taken from the last N months of counterparty metrics
TDH_TAB     = "TDH"       # the tab whose amounts are payment flows
OUT_DIR     = "dc_check_out"
REUSE_SAVED = True        # load saved results when the inputs are unchanged
TOP_N       = 15
NOT_FOUND_N = 40          # TDH links not found: rows shown (the full list is saved as a file)
FULL_LOAD_SHARE = 0.5     # customer fact: a load is "full" when it has at least this share of the date's largest load
DEFAULT_PD_KEYS = (39005, 39012)   # PD grades that mean defaulted (same as pkg_tm_cust_month)
DAILY_MONTHS = 24         # payment days read for the regularity look (the periodicity rule also looks back 24 months)
REG_MONTHS   = 13         # months of pkg_cust_reg_stream read (the latest row of each stream gives its state)
TIMELINE_N   = 24         # streams drawn in the payment-timeline figure

DEFAULT_DC_IDS = {"DCM9999999", "DCC9999999"}
REAL = "PNC Power ID"

if not Path(XLSX_PATH).exists():
    raise FileNotFoundError(f"Workbook not found at '{XLSX_PATH}'. Put it next to this notebook or fix XLSX_PATH.")
Path(OUT_DIR).mkdir(exist_ok=True)

# ── Spark session: inside the ACC limits (4g + 1g executors, 4g driver, max 512 executors) ──
spark = (SparkSession.builder
         .appName("pkg_datacenter_pkg_check")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.executor.cores", "1")
         .config("spark.executor.memoryOverhead", "1g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.dynamicAllocation.enabled", "true")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:                                                   # silence the Atlas lineage hook and executor-release noise
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
               "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
MD = StorageLevel.MEMORY_AND_DISK

# ── display: ONE consolidated HTML output per cell (long cells update the same output while they run) ──
_CSS = """<style>
.dcx h4{margin:14px 0 4px 0;font-family:sans-serif;font-size:13px}
.dcx table{border-collapse:collapse;font-size:12px;font-family:sans-serif}
.dcx th{background:#1f4e79;color:#fff;padding:3px 8px;text-align:left}
.dcx td{padding:2px 8px;border-bottom:1px solid #ddd;white-space:nowrap}
.dcx p{font-family:sans-serif;font-size:12px;margin:2px 0}
</style>"""

def _fmt(df, width=60):
    out = df.copy()
    for c in out.columns:
        s = out[c]
        if pd.api.types.is_bool_dtype(s):
            continue
        if pd.api.types.is_numeric_dtype(s):
            if str(c).endswith("%"):
                out[c] = s.map(lambda x: "" if pd.isna(x) else f"{x:.1%}")
            else:
                out[c] = s.map(lambda x: "" if pd.isna(x) else
                               (f"{x:,.0f}" if (abs(x) >= 100 or float(x).is_integer()) else f"{x:,.2f}"))
        else:
            out[c] = s.map(lambda x: "" if pd.isna(x) else (str(x) if len(str(x)) <= width else str(x)[:width - 1] + "…"))
    return out

def _render(blocks):
    parts = [_CSS, '<div class="dcx">']
    for title, obj in blocks:
        parts.append(f"<h4>{html.escape(str(title))}</h4>")
        if isinstance(obj, pd.DataFrame):
            parts.append("<p>(no rows)</p>" if obj.empty else _fmt(obj).to_html(index=False, border=0, na_rep=""))
        else:
            parts.append(f"<p>{html.escape(str(obj))}</p>")
    parts.append("</div>")
    return "".join(parts)

def show(*blocks):
    display(HTML(_render(blocks)))

class Status:
    """Progress lines while a cell runs; the same output is replaced by the result tables at the end."""
    def __init__(self, title):
        self.title, self.t0, self.lines = title, time.time(), []
        self.h = display(HTML(""), display_id=True)
        self.step("started")
    def step(self, msg):
        self.lines.append(f"{time.strftime('%H:%M:%S')} · {msg}")
        self.h.update(HTML(_render([(self.title + " — running", "")]) + "".join(
            f"<p style='font:12px sans-serif;margin:1px 0'>{html.escape(l)}</p>" for l in self.lines)))
    def done(self, *blocks):
        mins = (time.time() - self.t0) / 60
        self.h.update(HTML(_render(list(blocks) + [("Run time", f"{mins:.1f} min")])))

def cached(name, sig_obj, fn, st=None):
    """Run fn() once and keep its result on disk; reuse it while the inputs (sig_obj) are unchanged."""
    sig = hashlib.md5(json.dumps(sig_obj, sort_keys=True, default=str).encode()).hexdigest()
    p = Path(OUT_DIR) / f"_{name}.pkl"
    if REUSE_SAVED and p.exists():
        try:
            saved = pickle.loads(p.read_bytes())
            if saved.get("sig") == sig:
                if st: st.step(f"{name}: loaded saved result")
                return saved["data"]
        except Exception:
            pass
    if st: st.step(f"{name}: running on Spark …")
    data = fn()
    p.write_bytes(pickle.dumps({"sig": sig, "data": data}))
    n = sum(len(v) for v in data.values()) if isinstance(data, dict) else len(data)
    if st: st.step(f"{name}: done ({n:,} rows)")
    return data

# ── workbook parsers (same rules as pkg_datacenter_workbook_explore) ────────
def _txt(v):
    if v is None:
        return None
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float):
        if np.isnan(v):
            return None
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (datetime.datetime, datetime.date)):
        return v.isoformat()[:10]
    s = str(v).strip()
    return s if s else None

def norm(v):
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) else re.sub(r"\s+", " ", str(v)).strip().lower()

def clean_id(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = str(v).strip().upper()
    if not s:
        return np.nan
    return s.zfill(10) if (s.isdigit() and len(s) < 10) else s

def id_kind(s):
    if not isinstance(s, str):
        return "blank"
    if s in DEFAULT_DC_IDS:
        return "Default DC"
    if re.fullmatch(r"\d{10}", s):
        return REAL
    if s.startswith("DCM"):
        return "DCM"
    if s.startswith("DCC"):
        return "DCC"
    if s.startswith("XX"):
        return "XX"
    return "other"

def parse_amt(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = re.sub(r"[$,\s]", "", str(v))
    if s in ("", "-"):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

# Company-name key: upper case, brackets / punctuation / legal words removed, no spaces.
# The Python and the Spark version below apply the same steps, so keys are comparable.
_LEGAL = ["INC", "INCORPORATED", "LLC", "LP", "LLP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "PLC", "NA", "SA", "GMBH"]
_LEGAL_RX = " (?:" + "|".join(_LEGAL) + ")(?= )"
def name_key(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    s = re.sub(r"\(.*?\)", " ", str(v).upper())
    s = s.replace(".", "").replace("&", " AND ")
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = " " + re.sub(r" +", " ", s) + " "
    s = re.sub(_LEGAL_RX, "", s).replace(" ", "")
    return s if s else None

def sp_key(c):
    s = F.upper(c)
    s = F.regexp_replace(s, r"\(.*?\)", " ")
    s = F.regexp_replace(s, r"\.", "")
    s = F.regexp_replace(s, "&", " AND ")
    s = F.regexp_replace(s, "[^A-Z0-9 ]", " ")
    s = F.concat(F.lit(" "), F.regexp_replace(s, " +", " "), F.lit(" "))
    s = F.regexp_replace(F.regexp_replace(s, _LEGAL_RX, ""), " ", "")
    return F.when(F.length(s) > 0, s)

def ids_df(values, col):
    """Small one-column Spark DataFrame from a Python list (built without pandas)."""
    return spark.createDataFrame([(str(v),) for v in values], f"{col} string")

show(("Session", f"Spark {spark.version} · pandas {pd.__version__} · numpy {np.__version__} · results folder: {OUT_DIR}/"))

## 1. Pre-flight: tables, columns, months
Checks every table and column this notebook needs before any long step. If a required column is missing it stops here and lists what the table really has.

In [ ]:
# ── 1. Pre-flight ───────────────────────────────────────────────────────────
NEED = {   # table: (required columns, optional columns used when present)
    T_CUST_DIM: (["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name"], []),
    T_PAIR:     (["cust", "dir", "scope", "other", "amount", "n_txn", "month"], []),
    T_CUST_MET: (["mdm_id", "month"], ["amt_all_in", "amt_all_out"]),
    T_CPTY_MET: (["ent_id", "ent_name", "month"], ["eligible", "amt_in", "amt_out", "n_cust_all", "hub_tier", "name_class"]),
    T_TM:       (["customer_pwr_id", "month", "dhe"], ["pd_grade_now", "pd_pct_now", "is_default", "past_due_days", "watchlist_exposure"]),
    T_EDGE:     (["src_id", "src_kind", "dst_id", "dst_kind", "category", "amount", "n_txn", "month"], ["is_self"]),
    T_MAP:      (["cpty_id", "ent_l2"], []),
    T_EDGE_D:   (["txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "amount", "n_txn", "month"], ["is_self"]),
    T_REG:      (["cust", "dir", "other", "month", "cadence"],
                 ["status", "live", "due", "late", "skipped", "overdue", "stopped", "resumed", "last_dt", "next_dt", "med_gap"]),
    T_CFACT:    (["as_of_dt", "time_key", "cust_pwr_id", "dhe"], ["pd_key", "watchlist_exposure", "cust_past_due_days"]),
    T_PD:       (["pd_key", "pd_pct"], ["active_flg"]),
}
OPTIONAL = {T_EDGE: "rail check (section 8) is skipped", T_MAP: "rail check (section 8) is skipped",
            T_EDGE_D: "payment timelines and the on-the-fly regularity rule are skipped",
            T_REG: "stored periodicity is skipped (regularity comes from the on-the-fly rule only)",
            T_CFACT: "credit rows come from pkg_tm_cust_month only", T_PD: "no PD for customer-fact rows"}
def table_cols(t):
    try:
        return [r["col_name"] for r in spark.sql(f"DESCRIBE {t}").collect()
                if r["col_name"] and not r["col_name"].startswith("#")]
    except Exception as e:
        return None
def table_months(t):
    try:
        ps = [r[0] for r in spark.sql(f"SHOW PARTITIONS {t}").collect()]
        ms = sorted({p.split("month=")[1].split("/")[0] for p in ps if "month=" in p})
        if ms:
            return ms
    except Exception:
        pass
    return sorted(str(r[0]) for r in spark.table(t).select("month").distinct().collect())

HAVE, MONTHS, OK, rows, problems = {}, {}, {}, [], []
for t, (req, opt) in NEED.items():
    cols = table_cols(t)
    OK[t] = False
    if cols is None:
        if t in OPTIONAL:
            rows.append({"table": t.split(".")[1], "status": "not available: " + OPTIONAL[t]})
        else:
            problems.append(f"{t}: table not found or not readable")
            rows.append({"table": t.split(".")[1], "status": "NOT FOUND"})
        continue
    miss = [c for c in req if c not in cols]
    HAVE[t] = [c for c in opt if c in cols]
    MONTHS[t] = table_months(t) if "month" in cols else []
    OK[t] = not miss
    if miss and t not in OPTIONAL:
        problems.append(f"{t}: missing {miss}. Columns it has: {sorted(set(cols))}")
    rows.append({"table": t.split(".")[1], "status": "ok" if not miss else ("MISSING COLUMNS" if t not in OPTIONAL else "missing columns: " + OPTIONAL[t]),
                 "required missing": ", ".join(miss), "optional found": ", ".join(HAVE[t]),
                 "optional not found": ", ".join(c for c in opt if c not in cols),
                 "first month": MONTHS[t][0] if MONTHS[t] else "", "last month": MONTHS[t][-1] if MONTHS[t] else "",
                 "months": len(MONTHS[t])})
show(("1. Tables this notebook reads", pd.DataFrame(rows)))
if problems:
    raise RuntimeError("Pre-flight failed:\n" + "\n".join(problems))

PAIR_MONTH_LIST = [m for m in MONTHS[T_PAIR] if PAIR_MONTHS is None or PAIR_MONTHS[0] <= m <= PAIR_MONTHS[1]]
NAME_MONTH_LIST = MONTHS[T_CPTY_MET][-NAME_MONTHS:]
MET_MONTH_LIST  = MONTHS[T_CUST_MET][-12:]
TM_LAST         = MONTHS[T_TM][-1]

## 2. Workbook: ids, names and links
Reads the six relationship tabs (same reader as the first notebook) and lists what has to be looked up: real Power IDs, names without a Power ID, and the links between data centers and second-order parties.

In [ ]:
# ── 2. Workbook → nodes and links ───────────────────────────────────────────
def find_header(df, keys, max_rows=15):
    for i in range(min(max_rows, len(df))):
        cells = [norm(v) for v in df.iloc[i].tolist()]
        if all(any(k in c for c in cells) for k in keys):
            return i
    return None

WB, RAW = openpyxl.load_workbook(XLSX_PATH, data_only=True), {}
for ws in WB.worksheets:
    used = [(r, c) for (r, c), cell in getattr(ws, "_cells", {}).items() if cell.value is not None]
    nr, nc = (max(r for r, _ in used), max(c for _, c in used)) if used else (0, 0)
    rws = [[_txt(v) for v in r] for r in ws.iter_rows(min_row=1, max_row=nr, max_col=nc, values_only=True)] if nr else []
    RAW[ws.title] = pd.DataFrame(rws) if rws else pd.DataFrame()

FIELD_RULES = [("pnc_flag",    lambda h: "pnc flag" in h),
               ("master_id",   lambda h: "master power" in h),
               ("master_name", lambda h: "master name" in h),
               ("dc_id",       lambda h: h.startswith("datacenter customer power")),
               ("dc_name",     lambda h: h.startswith("datacenter customer name")),
               ("so_id",       lambda h: h.startswith("second order power")),
               ("so_name",     lambda h: h.startswith("second order customer name")),
               ("so_type",     lambda h: h.startswith("second order relationship type")),
               ("so_amount",   lambda h: h.startswith("second order loan amount"))]
frames = []
for tab, df in RAW.items():
    hdr = find_header(df, ["master power", "second order"])
    if hdr is None:
        continue
    heads = [norm(v) for v in df.iloc[hdr].tolist()]
    body = df.iloc[hdr + 1:].dropna(how="all")
    out = pd.DataFrame({"tab": tab, "excel_row": (body.index + 1).astype(str)}, index=body.index)
    for f, test in FIELD_RULES:
        j = next((j for j, h in enumerate(heads) if test(h)), None)
        out[f] = body.iloc[:, j] if j is not None else np.nan
    frames.append(out)
if not frames:
    raise RuntimeError("No relationship tab found in the workbook (looked for 'Master Power ID' + 'Second Order' headers).")
E = pd.concat(frames, ignore_index=True)
TABS = list(E.tab.unique())
for c in ("master_id", "dc_id", "so_id"):
    E[c] = E[c].map(clean_id)
for c in ("master_name", "dc_name", "so_name", "so_type"):
    E[c] = E[c].astype(object)
E["amount"] = E.so_amount.map(parse_amt)
E["type"]   = E.so_type.fillna("(blank)").astype(str).str.strip().str.title()

ROLES = [("Datacenter master", "master_id", "master_name"), ("Datacenter entity", "dc_id", "dc_name"),
         ("Second-order party", "so_id", "so_name")]
nd = []
for role, ic, nc_ in ROLES:
    d = E[[ic, nc_, "tab"]].dropna(subset=[ic]).rename(columns={ic: "id", nc_: "name"})
    d["role"] = role
    nd.append(d)
NODES = pd.concat(nd, ignore_index=True)
NODES["kind"] = NODES.id.map(id_kind)
NODES = NODES[NODES.kind != "Default DC"]
PIDS  = NODES[NODES.kind == REAL].copy()                       # real Power IDs → looked up by id
NAMES = NODES[NODES.kind != REAL].copy()                       # XX / DCM / DCC → looked up by name
NAMES["key"] = NAMES.name.map(name_key).map(lambda k: k if isinstance(k, str) and len(k) >= 3 else None)   # 1-2 letter keys are too weak to match on

# one row per distinct link: (tab, data center entity, second-order party, relationship type)
LK = (E.dropna(subset=["dc_id", "so_id"])
        .groupby(["tab", "master_id", "dc_id", "so_id", "type"], dropna=False, sort=False)
        .agg(master_name=("master_name", "first"), dc_name=("dc_name", "first"), so_name=("so_name", "first"),
             wb_amount=("amount", lambda s: s.sum(min_count=1)), wb_rows=("tab", "size")).reset_index())

cnt = []
for role, _, _ in ROLES:
    n = NODES[NODES.role == role].drop_duplicates("id")
    k = n.kind.value_counts()
    cnt.append({"role": role, "distinct ids": len(n), "real Power IDs (looked up by id)": int(k.get(REAL, 0)),
                "DCM / DCC (by name)": int(k.get("DCM", 0) + k.get("DCC", 0)), "XX (by name)": int(k.get("XX", 0)),
                "other": int(k.get("other", 0))})
lk = LK.groupby("tab", sort=False).agg(links=("so_id", "size"), with_amount=("wb_amount", "count")).reindex(TABS).reset_index()
show(("2a. What has to be found in PKG", pd.DataFrame(cnt)),
     ("2b. Links per tab (data center entity ↔ second-order party ↔ type)", lk.rename(columns={"with_amount": "with amount"})))

## 3. A. Clients: Power IDs in PKG
Each real Power ID is looked up in `pkg_cust_dim` both as a customer id and as a master (parent) id, then checked for payment metrics (last 12 months of `pkg_cust_metrics_monthly_v4`) and for a credit row in the latest month of `pkg_tm_cust_month` (DHE, PD).
A Power ID "found as master" brings its whole family of customers.

In [ ]:
# ── 3. A. Power IDs → pkg_cust_dim, customer metrics, TM / credit ───────────
st = Status("A. Power IDs in PKG")
pids = sorted(PIDS.id.unique())

def _idmap():
    b = F.broadcast(ids_df(pids, "pid"))
    cd = spark.table(T_CUST_DIM).select(F.col("mdm_id").cast("string").alias("mdm_id"),
                                        F.col("customer_pwr_id").cast("string").alias("customer_pwr_id"),
                                        F.col("master_pwr_id").cast("string").alias("master_pwr_id"), "customer_name")
    x = (cd.join(b, F.col("customer_pwr_id") == F.col("pid")).withColumn("level", F.lit("customer"))
           .unionByName(cd.join(b, F.col("master_pwr_id") == F.col("pid")).withColumn("level", F.lit("master"))))
    return x.toPandas()
IDMAP = cached("idmap", [T_CUST_DIM, pids], _idmap, st)
if IDMAP.empty:
    sample = [r[0] for r in spark.table(T_CUST_DIM).where(F.col("customer_pwr_id").isNotNull()).select("customer_pwr_id").limit(5).collect()]
    raise RuntimeError(f"No workbook Power ID matched pkg_cust_dim. Workbook ids look like {pids[:3]}; "
                       f"customer_pwr_id values look like {sample}. The id format differs: tell Claude before going on.")
mdm_all = sorted(IDMAP.mdm_id.dropna().unique())
cpw_all = sorted(set(IDMAP.customer_pwr_id.dropna()) | set(pids))      # also ids that are not in pkg_cust_dim

def _cmet():
    amt = [F.sum(F.col(c).cast("double")).alias(c) for c in HAVE[T_CUST_MET]]
    return (spark.table(T_CUST_MET).where(F.col("month").isin(MET_MONTH_LIST))
                 .select(F.col("mdm_id").cast("string").alias("mdm_id"), "month", *HAVE[T_CUST_MET])
                 .join(F.broadcast(ids_df(mdm_all, "mdm_id")), "mdm_id")
                 .groupBy("mdm_id").agg(F.count("*").alias("met_months"), F.max("month").alias("met_last"), *amt).toPandas())
CMET = cached("cust_metrics", [T_CUST_MET, MET_MONTH_LIST, mdm_all], _cmet, st)

def _tm():
    num = [c for c in HAVE[T_TM] if c in ("pd_pct_now", "past_due_days", "watchlist_exposure")]
    txt = [c for c in HAVE[T_TM] if c in ("pd_grade_now", "is_default")]
    nm = lambda c: c.replace("_now", "")                                   # pd_pct_now → pd_pct, pd_grade_now → pd_grade
    return (spark.table(T_TM).where(F.col("month") == TM_LAST)
                 .select(F.col("customer_pwr_id").cast("string").alias("customer_pwr_id"), F.col("dhe").cast("double").alias("dhe"),
                         *[F.col(c).cast("double").alias(nm(c)) for c in num], *[F.col(c).cast("string").alias(nm(c)) for c in txt])
                 .join(F.broadcast(ids_df(cpw_all, "customer_pwr_id")), "customer_pwr_id").toPandas())
TMX = cached("tm_credit_v2", [T_TM, TM_LAST, cpw_all, HAVE[T_TM]], _tm, st)

# Credit rows straight from the CDM customer fact for Power IDs that pkg_tm_cust_month does not hold (credit-only
# clients outside PKG). Same snapshot rule as pkg_tm_cust_month.ipynb: last as_of_dt of the month, its latest FULL load.
PID2CP = pd.concat([IDMAP[["pid", "customer_pwr_id"]], pd.DataFrame({"pid": pids, "customer_pwr_id": pids})]).dropna().drop_duplicates()
no_tm = sorted(set(pids) - set(PID2CP.merge(TMX[["customer_pwr_id"]], on="customer_pwr_id").pid))
CF_COLS = ["customer_pwr_id", "dhe", "pd_key", "pd_pct", "as_of_dt", "time_key"]
def _cf():
    y, mth = int(TM_LAST[:4]), int(TM_LAST[5:7])
    lo, hi = y * 10000 + mth * 100 + 1, y * 10000 + mth * 100 + 31
    ny, nm = (y + 1, 1) if mth == 12 else (y, mth + 1)
    f = (spark.table(T_CFACT).withColumn("_a", F.col("as_of_dt").cast("long")).withColumn("_t", F.col("time_key").cast("long"))
              .where(F.col("_a").between(lo, hi) & F.col("_t").between(lo, ny * 10000 + nm * 100 + 31)))
    fd = f.groupBy("_a", "_t").count().toPandas()
    if fd.empty:
        return pd.DataFrame(columns=CF_COLS)
    a = int(fd["_a"].max())
    ld = fd[fd["_a"] == a]
    tk = int(ld[ld["count"] >= FULL_LOAD_SHARE * ld["count"].max()]["_t"].max())
    pk = [F.col("pd_key").cast("long").alias("pd_key")] if "pd_key" in HAVE[T_CFACT] else [F.lit(None).cast("long").alias("pd_key")]
    x = (f.where((F.col("_a") == a) & (F.col("_t") == tk))
          .select(F.col("cust_pwr_id").cast("string").alias("customer_pwr_id"), F.col("dhe").cast("double").alias("dhe"), *pk)
          .join(F.broadcast(ids_df(no_tm, "customer_pwr_id")), "customer_pwr_id"))
    if OK.get(T_PD) and "pd_key" in HAVE[T_CFACT]:
        pdd = spark.table(T_PD)
        if "active_flg" in HAVE[T_PD]:
            pdd = pdd.where(F.upper(F.col("active_flg").cast("string")) == "Y")
        pdd = pdd.select(F.col("pd_key").cast("long").alias("pd_key"), F.col("pd_pct").cast("double").alias("pd_pct")).dropDuplicates(["pd_key"])
        x = x.join(F.broadcast(pdd), "pd_key", "left")
    else:
        x = x.withColumn("pd_pct", F.lit(None).cast("double"))
    out = x.toPandas()
    out["as_of_dt"], out["time_key"] = a, tk
    return out[CF_COLS]
CF = cached("cdm_credit", [T_CFACT, TM_LAST, no_tm, FULL_LOAD_SHARE], _cf, st) if (OK.get(T_CFACT) and no_tm) else pd.DataFrame(columns=CF_COLS)
CFG = (CF.assign(dflt=CF.pd_key.isin(DEFAULT_PD_KEYS)).groupby("customer_pwr_id")
         .agg(dhe=("dhe", "sum"), pd_pct=("pd_pct", "max"), dflt=("dflt", "max")))            # several rows per id: sum, worst PD

# one row per workbook Power ID
g = IDMAP.groupby("pid")
PID = pd.DataFrame({"pid": pids}).set_index("pid")
PID["as_customer"] = IDMAP[IDMAP.level == "customer"].groupby("pid").mdm_id.nunique()
PID["as_master"]   = IDMAP[IDMAP.level == "master"].groupby("pid").mdm_id.nunique()
PID["n_mdm"]       = g.mdm_id.nunique()
PID["pkg_name"]    = g.customer_name.first()
m = IDMAP[["pid", "mdm_id"]].drop_duplicates().merge(CMET, on="mdm_id")
gm = m.groupby("pid")
PID["mdm_with_metrics"] = gm.mdm_id.nunique()
PID["metrics_last_month"] = gm.met_last.max()
for c in HAVE[T_CUST_MET]:
    PID[c + "_12m"] = gm[c].sum()
t = PID2CP.merge(TMX, on="customer_pwr_id")
gt = t.groupby("pid")
PID["tm_customers"] = gt.customer_pwr_id.nunique()
PID["dhe"] = gt.dhe.sum(min_count=1)
if "pd_pct" in TMX:
    PID["pd_pct_max"] = gt.pd_pct.max()
    PID["has_pd"] = gt.pd_pct.max() > 0                                    # not rated / PD of 0 does not count
if "is_default" in TMX:
    PID["is_default"] = gt.is_default.apply(lambda s: s.astype(str).str.lower().isin(["1", "1.0", "true", "y", "yes"]).any())
for c in ("as_customer", "as_master", "n_mdm", "mdm_with_metrics", "tm_customers"):
    PID[c] = PID[c].fillna(0).astype(int)
PID["found"] = PID.n_mdm > 0
PID = PID.reset_index()
PID["credit_source"] = np.where(PID.tm_customers > 0, "pkg_tm_cust_month", "")
_cf_hit = (PID.tm_customers == 0) & PID.pid.isin(CFG.index)
PID.loc[_cf_hit, "credit_source"] = "cdm_customer_fact_v"
PID.loc[_cf_hit, "dhe"] = PID.loc[_cf_hit, "pid"].map(CFG.dhe)
for _c, _src in (("pd_pct_max", CFG.pd_pct), ("has_pd", CFG.pd_pct.fillna(0) > 0), ("is_default", CFG.dflt.astype(bool))):
    if _c not in PID:
        PID[_c] = np.nan
    PID[_c] = PID[_c].astype(object)
    PID.loc[_cf_hit, _c] = PID.loc[_cf_hit, "pid"].map(_src)
wb_name = PIDS.drop_duplicates("id").set_index("id").name
PID.insert(1, "workbook_name", PID.pid.map(wb_name))
PID.insert(2, "roles", PID.pid.map(PIDS.groupby("id").role.agg(lambda s: " + ".join(sorted(set(s))))))
PID.insert(3, "tabs", PID.pid.map(PIDS.groupby("id").tab.agg(lambda s: " + ".join(sorted(set(s))))))
PID.to_csv(Path(OUT_DIR) / "dc_power_ids_in_pkg.csv", index=False)

def _summ(d):
    n = len(d)
    r = {"Power IDs": n, "found in PKG": int(d.found.sum()), "found %": d.found.mean() if n else np.nan,
         "as customer id": int((d.as_customer > 0).sum()), "as master id": int((d.as_master > 0).sum()),
         "not found": int((~d.found).sum()), "mdm_ids": int(IDMAP[IDMAP.pid.isin(d.pid)].mdm_id.nunique()),
         "with payment metrics (12m)": int((d.mdm_with_metrics > 0).sum()),
         "with a credit row": int((d.credit_source != "").sum()),
         "of which from customer fact only": int((d.credit_source == "cdm_customer_fact_v").sum()),
         "credit row but not in pkg_cust_dim": int(((~d.found) & (d.credit_source != "")).sum()),
         "with DHE > 0": int((d.dhe.fillna(0) > 0).sum())}
    if "has_pd" in d:
        r["with a PD above 0"] = int(d.has_pd.fillna(False).astype(bool).sum())
    cp = PID2CP[PID2CP.pid.isin(d.pid)].customer_pwr_id.unique()
    r["DHE, each customer once"] = (TMX[TMX.customer_pwr_id.isin(cp)].dhe.sum()
                                    + d.loc[d.credit_source == "cdm_customer_fact_v", "dhe"].fillna(0).sum())
    return r
a1 = pd.DataFrame([{"role": role, **_summ(PID[PID.pid.isin(PIDS[PIDS.role == role].id)])} for role, _, _ in ROLES]
                  + [{"role": "ALL (distinct)", **_summ(PID)}])
so = PIDS[PIDS.role == "Second-order party"]
a2 = pd.DataFrame([{"tab": tab, **{k: v for k, v in _summ(PID[PID.pid.isin(so[so.tab == tab].id)]).items()
                                    if k in ("Power IDs", "found in PKG", "found %", "with payment metrics (12m)", "with DHE > 0", "DHE, each customer once")}}
                   for tab in TABS])
nf = PID[~PID.found][["pid", "workbook_name", "roles", "tabs", "credit_source", "dhe"]]
_cfnote = (f"customer fact as_of_dt {int(CF.as_of_dt.iloc[0])}, load {int(CF.time_key.iloc[0])}" if len(CF)
           else ("customer fact not needed" if not no_tm else "customer fact: no rows / not available"))
st.done(("A1. Workbook Power IDs found in pkg_cust_dim (customer metrics: last 12 months to "
         f"{MET_MONTH_LIST[-1]}; credit: pkg_tm_cust_month {TM_LAST}, then {_cfnote})", a1),
        ("A2. Second-order Power IDs by tab", a2),
        (f"A3. Power IDs not in pkg_cust_dim: {len(nf):,} (largest DHE first, first {TOP_N})",
         nf.sort_values("dhe", ascending=False, na_position="last").head(TOP_N)),
        ("Saved", f"{OUT_DIR}/dc_power_ids_in_pkg.csv: one row per Power ID with PKG name, mdm count, metrics, DHE and PD columns"))

## 4. B. Counterparties: workbook names in PKG
Every workbook name is matched to counterparty entities by a cleaned-name key (exact match on the key, so "Acme, Inc." = "ACME INC"; partial names are not matched). This is an inferred match.
Names of companies that have a Power ID are included too: a PNC client can still pay or be paid through an account at another bank, and PKG then sees it as an outside counterparty. For those, the customer names PKG holds for the same Power ID are used as well as the workbook name.
Tables B1 to B4 cover the names without a Power ID (XX / DCM / DCC ids).

In [ ]:
# ── 4. B. Non-PNC names → counterparty entities ─────────────────────────────
st = Status("B. Workbook names as PKG counterparties")
# Name keys for every node, not only the fake ids: a company that has a Power ID may still pay or be paid through an
# account at another bank, and then PKG sees it as an external counterparty, by name.
#   workbook name : the name typed in the workbook
#   PKG name      : customer names of the same Power ID in pkg_cust_dim (whole family for data centers, the customer itself for parties)
kr = []
for role, ic, nc_ in ROLES:
    d = E[[ic, nc_]].dropna(subset=[ic]).drop_duplicates()
    kr.append(pd.DataFrame({"role": role, "id": d[ic].values, "key": d[nc_].map(name_key).values, "src": "workbook name"}))
_ent_ids = set(E.dc_id.dropna()); _mas_ids = set(E.master_id.dropna()); _so_ids = set(E.so_id.dropna())
for role, sel in (("Datacenter entity", IDMAP[IDMAP.pid.isin(_ent_ids)]), ("Datacenter master", IDMAP[IDMAP.pid.isin(_mas_ids)]),
                  ("Second-order party", IDMAP[IDMAP.pid.isin(_so_ids) & (IDMAP.level == "customer")])):
    kr.append(pd.DataFrame({"role": role, "id": sel.pid.values, "key": sel.customer_name.map(name_key).values, "src": "PKG name"}))
KEYS = pd.concat(kr, ignore_index=True)
KEYS = KEYS[~KEYS.id.isin(DEFAULT_DC_IDS) & KEYS.key.map(lambda k: isinstance(k, str) and len(k) >= 3)]
KEYS = KEYS.sort_values("src", ascending=False).drop_duplicates(["role", "id", "key"]).reset_index(drop=True)   # workbook name wins a tie
keys = sorted(KEYS.key.unique())

def _ent():
    opt = HAVE[T_CPTY_MET]
    sel = [F.col("ent_id").cast("string").alias("ent_id"), "ent_name", "month"]
    sel += [F.col(c).cast("double").alias(c) for c in opt if c in ("amt_in", "amt_out", "n_cust_all", "hub_tier")]
    sel += [F.col(c).cast("string").alias(c) for c in opt if c in ("eligible", "name_class")]
    return (spark.table(T_CPTY_MET).where(F.col("month").isin(NAME_MONTH_LIST)).select(*sel)
                 .withColumn("key", sp_key(F.col("ent_name")))
                 .join(F.broadcast(ids_df(keys, "key")), "key").toPandas())
ENT = cached("cpty_entities", [T_CPTY_MET, NAME_MONTH_LIST, keys, _LEGAL], _ent, st)
for c in ("amt_in", "amt_out", "n_cust_all", "hub_tier"):
    if c not in ENT:
        ENT[c] = np.nan
ENT["eligible_flag"] = ENT["eligible"].astype(str).str.lower().isin(["1", "1.0", "true", "y", "yes"]) if "eligible" in ENT else False
last_nm = NAME_MONTH_LIST[-1]
ENTK = (ENT.sort_values("month").groupby(["key", "ent_id"])
           .agg(ent_name=("ent_name", "last"), months_active=("month", "nunique"), last_month=("month", "max"),
                amt_in=("amt_in", "sum"), amt_out=("amt_out", "sum"), n_cust=("n_cust_all", "max"),
                hub_tier=("hub_tier", "max"), full_metrics=("eligible_flag", "max")).reset_index())
KEYRES = ENTK.groupby("key").agg(entities=("ent_id", "nunique"), amt_in=("amt_in", "sum"), amt_out=("amt_out", "sum"),
                                 n_cust=("n_cust", "max"), full_metrics=("full_metrics", "max"),
                                 pkg_names=("ent_name", lambda s: " / ".join(s.drop_duplicates().head(2)))).reset_index()
NM = NAMES.drop_duplicates(["id", "role"]).merge(KEYRES, on="key", how="left")
NM["matched"] = NM.entities.fillna(0) > 0
NM.to_csv(Path(OUT_DIR) / "dc_names_in_pkg.csv", index=False)

def _nsum(d):
    n = len(d)
    return {"names": n, "seen as counterparty": int(d.matched.sum()), "seen %": d.matched.mean() if n else np.nan,
            "PKG entities": int(d.entities.fillna(0).sum()), "with full counterparty metrics": int(d.full_metrics.fillna(False).astype(bool).sum()),
            "dollars with PNC customers": d.amt_in.fillna(0).sum() + d.amt_out.fillna(0).sum()}
b1 = pd.DataFrame([{"role": role, "id type": k, **_nsum(NM[(NM.role == role) & (NM.kind == k)])}
                   for role, _, _ in ROLES for k in ("DCM", "DCC", "XX", "other") if ((NM.role == role) & (NM.kind == k)).any()])
b2 = pd.DataFrame([{"tab": tab, **_nsum(NAMES[NAMES.tab == tab].drop_duplicates("id").merge(KEYRES, on="key", how="left")
                                         .assign(matched=lambda d: d.entities.fillna(0) > 0))} for tab in TABS])
top = (NM[NM.matched].drop_duplicates("id").assign(usd=lambda d: d.amt_in.fillna(0) + d.amt_out.fillna(0))
         .sort_values("usd", ascending=False)[["id", "name", "role", "pkg_names", "entities", "n_cust", "usd"]].head(TOP_N)
         .rename(columns={"pkg_names": "name(s) in PKG", "n_cust": "PNC customers (max)", "usd": f"dollars, {len(NAME_MONTH_LIST)} months"}))
dcu = NM[(~NM.matched) & (NM.role != "Second-order party")].drop_duplicates("id")[["id", "name", "role", "key"]]
_real = KEYS[KEYS.id.map(id_kind) == REAL].assign(hit=lambda d: d.key.isin(set(ENT.key)))
b0 = (_real.groupby(["role", "id"]).hit.max().reset_index().groupby("role")
           .agg(power_ids=("id", "size"), also_seen_by_name=("hit", "sum")).reset_index())
b0["seen %"] = b0.also_seen_by_name / b0.power_ids
b0 = b0.rename(columns={"power_ids": "Power IDs", "also_seen_by_name": "also seen as an external counterparty (by name)"})
st.done(("B0. Companies WITH a Power ID that PKG also sees as an external counterparty (accounts at other banks; inferred, by name key)", b0),
        (f"B1. Names without a Power ID seen as counterparty entities ({NAME_MONTH_LIST[0]} to {last_nm}; inferred, by name key)", b1),
        ("B2. By tab", b2),
        (f"B3. Largest matched names (top {TOP_N})", top),
        (f"B4. Non-PNC data centers NOT seen as counterparties: {len(dcu):,} (first {TOP_N})", dcu.head(TOP_N)),
        ("Saved", f"{OUT_DIR}/dc_names_in_pkg.csv: one row per non-PNC name with its match and PKG dollars"))

## 5. C. Pair-table scan
One pass over `pkg_cust_pair_month_v4`, keeping only (1) every relationship of the data centers that are PNC customers and (2) relationships between second-order PNC customers and outside counterparty entities named like a data center. This is the long step.

In [ ]:
# ── 5. C. Scan the pair table once ──────────────────────────────────────────
st = Status("C. Pair-table scan")
ent_real = sorted(set(E.dc_id.dropna()[E.dc_id.dropna().map(id_kind) == REAL]))
mas_real = sorted(set(E.master_id.dropna()[E.master_id.dropna().map(id_kind) == REAL]))
so_real  = sorted(set(E.so_id.dropna()[E.so_id.dropna().map(id_kind) == REAL]))
dc_mdm    = sorted(IDMAP[IDMAP.pid.isin(set(ent_real) | set(mas_real))].mdm_id.unique())
party_mdm = sorted(IDMAP[IDMAP.pid.isin(so_real)].mdm_id.unique())
dc_keys   = set(KEYS[KEYS.role != "Second-order party"].key)
so_keys   = set(KEYS[KEYS.role == "Second-order party"].key)
dc_ent    = sorted(ENT[ENT.key.isin(dc_keys)].ent_id.unique())
party_ent = sorted(ENT[ENT.key.isin(so_keys)].ent_id.unique())
st.step(f"{len(PAIR_MONTH_LIST)} months ({PAIR_MONTH_LIST[0]} to {PAIR_MONTH_LIST[-1]}) · {len(dc_mdm):,} data center mdm_ids · "
        f"{len(party_mdm):,} second-order mdm_ids · {len(dc_ent):,} entities named like a data center · "
        f"{len(party_ent):,} entities named like a second-order party")

def _scan():
    pr = (spark.table(T_PAIR).where(F.col("month").isin(PAIR_MONTH_LIST) & F.col("scope").isin("int", "ext"))
               .select(F.col("cust").cast("string").alias("cust"), "dir", "scope", F.col("other").cast("string").alias("other"),
                       F.col("amount").cast("double").alias("amount"), F.col("n_txn").cast("long").alias("n_txn"), "month"))
    T = F.lit(True)
    pr = pr.join(F.broadcast(ids_df(sorted(set(dc_mdm) | set(party_mdm)), "cust")), "cust")      # keep only our customers' rows
    a = (pr.join(F.broadcast(ids_df(dc_mdm, "cust").withColumn("is_dc", T)), "cust", "left")
           .join(F.broadcast(ids_df(party_mdm, "cust").withColumn("is_party", T)), "cust", "left")
           .join(F.broadcast(ids_df(dc_ent, "other").withColumn("o_dcent", T)), "other", "left")
           .join(F.broadcast(ids_df(party_mdm, "other").withColumn("o_party", T)), "other", "left")
           .join(F.broadcast(ids_df(party_ent, "other").withColumn("o_partyent", T)), "other", "left"))
    for c in ("is_dc", "is_party", "o_dcent", "o_party", "o_partyent"):
        a = a.withColumn(c, F.coalesce(F.col(c), F.lit(False)))
    hits = a.where(F.col("is_dc") | (F.col("is_party") & F.col("o_dcent") & (F.col("scope") == "ext"))).persist(MD)
    n = hits.count()
    st.step(f"scan finished: {n:,} pair-month rows kept")
    nbr = (hits.where("is_dc").groupBy("cust", "dir", "scope", "other")
               .agg(F.sum("amount").alias("amount"), F.sum("n_txn").alias("n_txn"), F.count("*").alias("n_months"),
                    F.min("month").alias("first_month"), F.max("month").alias("last_month")).toPandas())
    int_m = (hits.where(F.col("is_dc") & (F.col("scope") == "int") & F.col("o_party"))
                 .select("cust", "other", "dir", "month", "amount", "n_txn").toPandas())
    party_m = (hits.where(F.col("is_party") & F.col("o_dcent") & (F.col("scope") == "ext"))
                   .select("cust", "other", "dir", "month", "amount", "n_txn").toPandas())
    ext_m = (hits.where(F.col("is_dc") & (F.col("scope") == "ext") & F.col("o_partyent"))
                 .select("cust", "other", "dir", "month", "amount", "n_txn").toPandas())
    hits.unpersist()
    return {"nbr": nbr, "int_m": int_m, "party_m": party_m, "ext_m": ext_m}
SCAN = cached("pair_scan_v2", [T_PAIR, PAIR_MONTH_LIST, dc_mdm, party_mdm, dc_ent, party_ent], _scan, st)
NBR, INT_M, PARTY_M, EXT_M = SCAN["nbr"], SCAN["int_m"], SCAN["party_m"], SCAN["ext_m"]
for d in (NBR, INT_M, PARTY_M, EXT_M):
    for c in ("cust", "other", "dir"):
        d[c] = d[c].astype(object)

dcn = NBR.groupby("cust").agg(neighbours=("other", "nunique"), dollars=("amount", "sum"))
c1 = pd.DataFrame([
    {"item": "data center mdm_ids with at least one payment relationship", "value": f"{len(dcn):,} of {len(dc_mdm):,}"},
    {"item": "relationships of those data centers (customer × direction × counterparty)", "value": f"{len(NBR):,}"},
    {"item": "   with other PNC customers (scope int)", "value": f"{int((NBR.scope == 'int').sum()):,}"},
    {"item": "   with external counterparty entities (scope ext)", "value": f"{int((NBR.scope == 'ext').sum()):,}"},
    {"item": "dollars on those relationships, all months", "value": f"{NBR.amount.sum():,.0f}"},
    {"item": "monthly rows, route 1: data center customer ↔ second-order customer (both bank here)", "value": f"{len(INT_M):,}"},
    {"item": "monthly rows, route 2: data center customer ↔ outside account named like the party", "value": f"{len(EXT_M):,}"},
    {"item": "monthly rows, route 3: second-order customer ↔ outside account named like the data center", "value": f"{len(PARTY_M):,}"},
])
st.done(("C1. What the scan kept", c1))

## 6. C. Are the workbook links payment relationships in PKG?
A payment between two companies can reach PKG by three routes: both bank at PNC (matched by Power IDs), only the data center banks at PNC (the party is matched by name), or only the party banks at PNC (the data center is matched by name). All three are tried for every link.
Each link is tested at two levels: the data center entity itself, and its parent's whole family (when the entity shows nothing).
Direction is read from the data center's side: a Customer should pay the data center, a Supplier should be paid by it.

In [ ]:
# ── 6. C. Workbook links vs PKG payment relationships ───────────────────────
COLS = ["dc_key", "so_id", "dir", "amount", "n_txn", "first_month", "last_month", "basis"]
so_map  = (IDMAP[IDMAP.pid.isin(so_real)][["pid", "mdm_id"]].drop_duplicates()
           .rename(columns={"pid": "so_id", "mdm_id": "other"}))
ent2key = ENT[["ent_id", "key"]].drop_duplicates().rename(columns={"ent_id": "other"})
so_nm   = KEYS[KEYS.role == "Second-order party"][["id", "key"]].drop_duplicates().rename(columns={"id": "so_id"})
dc_e    = IDMAP[IDMAP.pid.isin(ent_real)][["pid", "mdm_id"]].drop_duplicates().rename(columns={"pid": "dc_key", "mdm_id": "cust"})
dc_p    = IDMAP[IDMAP.pid.isin(mas_real)][["pid", "mdm_id"]].drop_duplicates().rename(columns={"pid": "dc_key", "mdm_id": "cust"})
dcn_e   = KEYS[KEYS.role == "Datacenter entity"][["id", "key"]].drop_duplicates().rename(columns={"id": "dc_key"})
dcn_p   = KEYS[KEYS.role == "Datacenter master"][["id", "key"]].drop_duplicates().rename(columns={"id": "dc_key"})

# Three routes by which PKG can see a payment between a data center and a second-order party:
#   route 1 (ids)              both bank at PNC: customer ↔ customer
#   route 2 (party name)       the data center banks at PNC, the party is paid / pays through an account elsewhere
#   route 3 (data center name) the party banks at PNC, the data center pays / is paid through an account elsewhere
# The three are different accounts, so their dollars add up without double counting.
F1 = NBR[NBR.scope == "int"].merge(so_map, on="other").assign(basis="ids")
F2 = NBR[NBR.scope == "ext"].merge(ent2key, on="other").merge(so_nm, on="key").assign(basis="party name")
FA = pd.concat([F1, F2], ignore_index=True)
# route 3 is seen from the second-order customer's side and flipped to the data center's view
PM = PARTY_M.merge(so_map.rename(columns={"other": "cust"}), on="cust").merge(ent2key, on="other")
PM["dir"] = PM.dir.map({"in": "out", "out": "in"})

def roll(dcmap, dcnames):
    a = (FA.merge(dcmap, on="cust").groupby(["dc_key", "so_id", "dir", "basis"])
           .agg(amount=("amount", "sum"), n_txn=("n_txn", "sum"), first_month=("first_month", "min"),
                last_month=("last_month", "max")).reset_index())
    b = (PM.merge(dcnames, on="key").groupby(["dc_key", "so_id", "dir"])
           .agg(amount=("amount", "sum"), n_txn=("n_txn", "sum"), first_month=("month", "min"),
                last_month=("month", "max")).reset_index().assign(basis="data center name"))
    return pd.concat([a[COLS], b[COLS]], ignore_index=True)

def wide(fl, key, sfx):
    cols = [key, "so_id", "pkg_in" + sfx, "pkg_out" + sfx, "n_txn" + sfx, "first_month" + sfx, "last_month" + sfx, "basis" + sfx]
    if fl.empty:
        return pd.DataFrame(columns=cols)
    g = fl.groupby(["dc_key", "so_id"])
    w = pd.DataFrame({"pkg_in" + sfx: fl[fl.dir == "in"].groupby(["dc_key", "so_id"]).amount.sum(),
                      "pkg_out" + sfx: fl[fl.dir == "out"].groupby(["dc_key", "so_id"]).amount.sum()})
    w = w.reindex(g.size().index)
    w["n_txn" + sfx] = g.n_txn.sum()
    w["first_month" + sfx] = g.first_month.min()
    w["last_month" + sfx] = g.last_month.max()
    w["basis" + sfx] = g.basis.agg(lambda s: " + ".join(sorted(set(s))))
    return w.reset_index().rename(columns={"dc_key": key})[cols]

FLOW_E, FLOW_P = roll(dc_e, dcn_e), roll(dc_p, dcn_p)
L = (LK.merge(wide(FLOW_E, "dc_id", ""), on=["dc_id", "so_id"], how="left")
       .merge(wide(FLOW_P, "master_id", "_par"), on=["master_id", "so_id"], how="left"))
for c in ("pkg_in", "pkg_out", "pkg_in_par", "pkg_out_par"):
    L[c] = pd.to_numeric(L[c], errors="coerce").fillna(0.0)

found_pid, ent_keys = set(IDMAP.pid), set(ENT.key)
_seen = KEYS[KEYS.key.isin(ent_keys)]
seen_e, seen_p = set(_seen[_seen.role == "Datacenter entity"].id), set(_seen[_seen.role == "Datacenter master"].id)
seen_so = set(_seen[_seen.role == "Second-order party"].id)
L["dc_cust"] = [(a in found_pid) or (b in found_pid) for a, b in zip(L.dc_id, L.master_id)]     # data center banks at PNC (in PKG as a customer)
L["so_cust"] = L.so_id.isin(found_pid)                                                           # party banks at PNC
L["dc_seen"] = [(a in seen_e) or (b in seen_p) for a, b in zip(L.dc_id, L.master_id)]            # data center name seen as an outside counterparty
L["so_seen"] = L.so_id.isin(seen_so)
L["f_ent"]   = (L.pkg_in + L.pkg_out) > 0
L["f_par"]   = (L.pkg_in_par + L.pkg_out_par) > 0

def _status(r):
    if r.dc_id in DEFAULT_DC_IDS:        return "0 Default DC bucket (no specific data center)"
    if not r.dc_cust and not r.so_cust:  return "1 neither side banks at PNC (PKG cannot see it)"
    if not ((r.dc_cust and (r.so_cust or r.so_seen)) or (r.so_cust and r.dc_seen)):
        return "3 second-order party not seen in PKG (id or name)" if r.dc_cust else "2 data center not seen in PKG (id or name)"
    if r.f_ent:                          return "5 payments found"
    if r.f_par:                          return "6 payments found via the parent's family only"
    return "4 both sides seen, no payments between them"
L["status"] = L.apply(_status, axis=1)
L["testable"] = L.status.str[0].isin(["4", "5", "6"])
L["found"] = L.status.str[0].isin(["5", "6"])
use_par = L.status.str[0] == "6"
L["m_in"], L["m_out"] = np.where(use_par, L.pkg_in_par, L.pkg_in), np.where(use_par, L.pkg_out_par, L.pkg_out)
L["m_last"]  = np.where(use_par, L.last_month_par, L.last_month)
L["m_basis"] = np.where(use_par, L.basis_par, L.basis)
exp = L.type.map({"Customer": "in", "Supplier": "out"})
L["dir_ok"] = np.where(exp == "in", L.m_in > 0, np.where(exp == "out", L.m_out > 0, np.nan))
L.loc[~L.found | exp.isna(), "dir_ok"] = np.nan

(L.drop(columns=["f_ent", "f_par"])
  .to_csv(Path(OUT_DIR) / "dc_links_checked.csv", index=False))

def _row(d):
    t, f = int(d.testable.sum()), int(d.found.sum())
    dk = d.dir_ok.dropna()
    return {"links": len(d), "testable": t, "payments found": f, "found %": f / t if t else np.nan,
            "of which via parent only": int((d.status.str[0] == "6").sum()),
            "by ids only (observed)": int((d.found & (d.m_basis == "ids")).sum()),
            "by name, fully or partly (inferred)": int((d.found & (d.m_basis != "ids")).sum()),
            "stated direction %": dk.mean() if len(dk) else np.nan}
c2 = pd.DataFrame([{"tab": tab, **_row(L[L.tab == tab])} for tab in TABS] + [{"tab": "ALL TABS", **_row(L)}])
c3 = L.groupby(["status", "tab"]).size().unstack(fill_value=0).reindex(columns=TABS, fill_value=0).rename_axis(None, axis=1)
c3.insert(0, "status", [i[2:] for i in c3.index])
c3["ALL"] = c3[TABS].sum(axis=1)
c4 = (L[L.found].groupby("type").agg(links=("so_id", "size"), pkg_in=("m_in", "sum"), pkg_out=("m_out", "sum")).reset_index()
        .rename(columns={"type": "relationship type", "pkg_in": "PKG dollars paid TO the data center", "pkg_out": "PKG dollars paid BY the data center"}))
show((f"C2. Workbook links that are direct payment relationships in PKG ({PAIR_MONTH_LIST[0]} to {PAIR_MONTH_LIST[-1]})", c2),
     ("C3. Status of every link, by tab", c3.reset_index(drop=True)),
     ("C4. Found links by relationship type: which way the money goes in PKG", c4),
     ("Saved", f"{OUT_DIR}/dc_links_checked.csv: every workbook link with its PKG dollars in / out, last month, match basis and status"))

## 7. C. TDH amounts against PKG dollars
TDH does not say which period its amounts cover, so PKG dollars are summed over many candidate periods (calendar years, and 1 / 3 / 6 / 12-month windows ending in each recent month) and the best-fitting periods are listed.
All three routes are included in the PKG dollars.

In [ ]:
# ── 7. C. TDH amounts vs PKG dollars ────────────────────────────────────────
T = L[(L.tab == TDH_TAB) & L.wb_amount.notna() & (L.wb_amount > 0)].copy()
T["exp"] = T.type.map({"Customer": "in", "Supplier": "out"})
tdh_tot = T.wb_amount.sum()
d1 = pd.DataFrame([{"TDH links with an amount": len(T), "TDH dollars": tdh_tot,
                    "links found in PKG": int(T.found.sum()), "found %": T.found.mean() if len(T) else np.nan,
                    "TDH dollars on found links": T.loc[T.found, "wb_amount"].sum(),
                    "dollar %": T.loc[T.found, "wb_amount"].sum() / tdh_tot if tdh_tot else np.nan,
                    "found in the stated direction %": T.dir_ok.dropna().mean() if T.dir_ok.notna().any() else np.nan}])
d1b = (T.assign(dc=np.where(T.dc_id.map(id_kind) == REAL, "data center has a Power ID", "data center without a Power ID"))
         .groupby(["dc", "type"]).agg(links=("so_id", "size"), found=("found", "sum"), tdh=("wb_amount", "sum"),
                                      tdh_found=("wb_amount", lambda s: s[T.loc[s.index, "found"]].sum())).reset_index())
d1b["found %"] = d1b.found / d1b.links
d1b["dollar %"] = d1b.tdh_found / d1b.tdh
d1b = d1b.rename(columns={"dc": "data center", "type": "relationship type", "tdh": "TDH dollars", "tdh_found": "TDH dollars found"})

# monthly PKG dollars per TDH link, in the stated direction, at the level the link was found
IM = pd.concat([INT_M.merge(so_map, on="other")[["cust", "so_id", "dir", "month", "amount"]],
                EXT_M.merge(ent2key, on="other").merge(so_nm, on="key")[["cust", "so_id", "dir", "month", "amount"]]], ignore_index=True)
def monthly(dcmap, dcnames):
    a = IM.merge(dcmap, on="cust")[["dc_key", "so_id", "dir", "month", "amount"]]
    b = PM.merge(dcnames, on="key")[["dc_key", "so_id", "dir", "month", "amount"]]
    return pd.concat([a, b], ignore_index=True).groupby(["dc_key", "so_id", "dir", "month"], as_index=False).amount.sum()
ME, MP = monthly(dc_e, dcn_e), monthly(dc_p, dcn_p)
Tf = T[T.found & T.exp.notna()].reset_index(drop=True)
Tf["lid"] = Tf.index
se = Tf[Tf.status.str[0] == "5"].merge(ME, left_on=["dc_id", "so_id", "exp"], right_on=["dc_key", "so_id", "dir"])
sp = Tf[Tf.status.str[0] == "6"].merge(MP, left_on=["master_id", "so_id", "exp"], right_on=["dc_key", "so_id", "dir"])
SER = pd.concat([se[["lid", "month", "amount"]], sp[["lid", "month", "amount"]]], ignore_index=True)
M = (SER.pivot_table(index="lid", columns="month", values="amount", aggfunc="sum")
        .reindex(index=Tf.lid, columns=PAIR_MONTH_LIST).fillna(0.0)) if len(SER) else pd.DataFrame(0.0, index=Tf.lid, columns=PAIR_MONTH_LIST)

wins = [("all months", PAIR_MONTH_LIST)]
for y in sorted({m[:4] for m in PAIR_MONTH_LIST}):
    wins.append((f"calendar {y}", [m for m in PAIR_MONTH_LIST if m[:4] == y]))
for end in range(len(PAIR_MONTH_LIST) - 1, max(len(PAIR_MONTH_LIST) - 19, -1), -1):
    for k in (12, 6, 3, 1):
        if end - k + 1 >= 0:
            wins.append((f"{k} months to {PAIR_MONTH_LIST[end]}", PAIR_MONTH_LIST[end - k + 1:end + 1]))
wr, tdh = [], Tf.wb_amount.to_numpy(dtype=float)
for name, ms in wins:
    pk = M[ms].sum(axis=1).to_numpy(dtype=float)
    has = pk > 0
    ratio = pk[has] / tdh[has]
    wr.append({"PKG period": name, "months": len(ms), "links with PKG dollars": int(has.sum()),
               "median PKG / TDH": float(np.median(ratio)) if has.any() else np.nan,
               "within 1%": float((np.abs(ratio - 1) <= 0.01).sum() / len(tdh)) if len(tdh) else np.nan,
               "within 10%": float((np.abs(ratio - 1) <= 0.10).sum() / len(tdh)) if len(tdh) else np.nan,
               "within 25%": float((np.abs(ratio - 1) <= 0.25).sum() / len(tdh)) if len(tdh) else np.nan,
               "PKG total / TDH total": pk.sum() / tdh.sum() if tdh.sum() else np.nan})
WR = pd.DataFrame(wr).sort_values(["within 10%", "within 25%"], ascending=False).reset_index(drop=True)
WR = WR.rename(columns={"within 1%": "exact (±1%) %", "within 10%": "within ±10% %", "within 25%": "within ±25% %"})
best = WR.iloc[0]["PKG period"] if len(WR) and len(Tf) else None
d1c = (T[T.found].groupby("m_basis").agg(links=("so_id", "size"), tdh=("wb_amount", "sum")).reset_index()
          .rename(columns={"m_basis": "matched by", "tdh": "TDH dollars"}))
blocks = [("D1. TDH links and dollars found in PKG", d1), ("D2. By kind of data center and relationship type", d1b),
          ("D2b. Found TDH links by how they were matched (ids = route 1, party name = route 2, data center name = route 3)", d1c),
          (f"D3. Which PKG period fits the TDH amounts best ({len(Tf):,} found links; shares are of those links; top 8 of {len(WR)} periods)", WR.head(8))]
if best:
    ms = dict(wins)[best]
    Tf["pkg_best"] = M[ms].sum(axis=1).to_numpy()
    Tf["ratio"] = Tf.pkg_best / Tf.wb_amount
    Tf["gap"] = (np.log(Tf.ratio.where(Tf.ratio > 0))).abs()
    ex = (Tf.sort_values("wb_amount", ascending=False)[["dc_name", "so_name", "type", "wb_amount", "pkg_best", "ratio", "m_basis"]].head(TOP_N)
            .rename(columns={"dc_name": "data center", "so_name": "second-order party", "wb_amount": "TDH amount",
                             "pkg_best": f"PKG, {best}", "ratio": "PKG / TDH", "m_basis": "matched by"}))
    blocks.append((f"D4. Largest TDH links next to PKG dollars for the best period ({best})", ex))
    Tf[["tab", "master_id", "dc_id", "so_id", "type", "dc_name", "so_name", "wb_amount", "pkg_best", "ratio", "status", "m_basis"]] \
        .to_csv(Path(OUT_DIR) / "dc_tdh_vs_pkg.csv", index=False)
# every TDH link not found: what was tried, saved in full for investigation
_kd = {role: d.groupby("id").key.agg(lambda x: " | ".join(sorted(set(x)))) for role, d in KEYS.groupby("role")}
_k = lambda role, ids: ids.map(_kd.get(role, pd.Series(dtype=object))).fillna("")
_pn = PID.set_index("pid").pkg_name
_yn = lambda b: np.where(b, "yes", "")
NF = T[~T.found].sort_values("wb_amount", ascending=False).copy()
NF["status"] = NF.status.str[2:]
NF["dc_banks_at_pnc"], NF["party_banks_at_pnc"] = _yn(NF.dc_cust), _yn(NF.so_cust)
NF["dc_name_seen_outside"], NF["party_name_seen_outside"] = _yn(NF.dc_seen), _yn(NF.so_seen)
NF["dc_pkg_name"] = NF.dc_id.map(_pn).fillna(NF.master_id.map(_pn))
NF["party_pkg_name"] = NF.so_id.map(_pn)
NF["dc_name_keys_tried"] = (_k("Datacenter entity", NF.dc_id) + " | " + _k("Datacenter master", NF.master_id)).str.strip(" |")
NF["party_name_keys_tried"] = _k("Second-order party", NF.so_id)
NF[["master_id", "master_name", "dc_id", "dc_name", "so_id", "so_name", "type", "wb_amount", "status", "dc_banks_at_pnc", "party_banks_at_pnc",
    "dc_name_seen_outside", "party_name_seen_outside", "dc_pkg_name", "party_pkg_name", "dc_name_keys_tried", "party_name_keys_tried"]] \
    .to_csv(Path(OUT_DIR) / "dc_tdh_not_found.csv", index=False)
miss = (NF[["master_name", "so_name", "so_id", "type", "wb_amount", "status", "dc_banks_at_pnc", "party_banks_at_pnc",
            "dc_name_seen_outside", "party_name_seen_outside"]].head(NOT_FOUND_N)
          .rename(columns={"master_name": "data center (parent)", "so_name": "second-order party", "so_id": "party id", "wb_amount": "TDH amount",
                           "dc_banks_at_pnc": "DC banks here", "party_banks_at_pnc": "party banks here",
                           "dc_name_seen_outside": "DC name seen", "party_name_seen_outside": "party name seen"}))
byp = (NF.groupby("master_name", dropna=False).agg(links=("so_id", "size"), tdh=("wb_amount", "sum"),
                                                     dc_here=("dc_cust", "max"), dc_seen=("dc_seen", "max"),
                                                     keys=("dc_name_keys_tried", "first")).reset_index()
         .sort_values("tdh", ascending=False).head(20))
byp["dc_here"], byp["dc_seen"] = _yn(byp.dc_here.astype(bool)), _yn(byp.dc_seen.astype(bool))
byp = byp.rename(columns={"master_name": "data center (parent)", "links": "links not found", "tdh": "TDH dollars not found",
                          "dc_here": "DC banks here", "dc_seen": "DC name seen", "keys": "name keys tried for the data center"})
blocks.append((f"D5. TDH links NOT found in PKG: {len(NF):,} links, {NF.wb_amount.sum():,.0f} dollars (largest {NOT_FOUND_N}; "
               f"full list with ids and the name keys tried: {OUT_DIR}/dc_tdh_not_found.csv)", miss))
blocks.append(("D6. Not-found TDH links by data center (top 20 by dollars)", byp))
show(*blocks)

## 8. C. Rails: is TDH only part of what we hold?
For the found TDH links and the best-fitting period, PKG dollars are split by payment rail from `pkg_edge_monthly` (the pair table has no rail). If TDH holds ACH only, "ACH only" should match the TDH amounts better than "all rails", and the other rails are payments we have that the workbook does not.

In [ ]:
# ── 8. C. Rails behind the found TDH links ──────────────────────────────────
st = Status("Rails behind the TDH links")
RAILS = ["ACH", "WIRE", "RTP", "CHECK", "PCARD", "OTHER"]
TR = None
if not (OK.get(T_EDGE) and OK.get(T_MAP)):
    st.done(("8. Rails", "Skipped: pkg_edge_monthly or pkg_cpty_entity_map_v2 is not available (see pre-flight)."))
elif not best or not len(Tf):
    st.done(("8. Rails", "Skipped: no TDH link was found, so there is nothing to split by rail."))
else:
    ms = [m for m in dict(wins)[best] if m in MONTHS[T_EDGE]]
    # the customer ↔ counterparty pairs behind every found TDH link (route 1: other = mdm_id; routes 2 and 3: other = entity)
    PC = ["cust", "other", "so_id", "okind", "flip"]
    r1 = INT_M[["cust", "other"]].drop_duplicates().merge(so_map, on="other").assign(okind="CUST", flip=False)[PC]
    r2 = (EXT_M[["cust", "other"]].drop_duplicates().merge(ent2key, on="other").merge(so_nm, on="key")
          .assign(okind="CPTY", flip=False)[PC].drop_duplicates())
    r3 = PM[["cust", "other", "so_id", "key"]].drop_duplicates().assign(okind="CPTY", flip=True)
    def pairs(dcmap, dcnames):
        a = pd.concat([r1, r2], ignore_index=True).merge(dcmap, on="cust")
        b = r3.merge(dcnames, on="key")
        return pd.concat([a, b], ignore_index=True)[["dc_key"] + PC].drop_duplicates()
    PE, PP = pairs(dc_e, dcn_e), pairs(dc_p, dcn_p)
    pt = pd.concat([Tf[Tf.status.str[0] == "5"][["lid", "dc_id", "so_id", "exp"]].merge(PE, left_on=["dc_id", "so_id"], right_on=["dc_key", "so_id"]),
                    Tf[Tf.status.str[0] == "6"][["lid", "master_id", "so_id", "exp"]].merge(PP, left_on=["master_id", "so_id"], right_on=["dc_key", "so_id"])],
                   ignore_index=True)[["lid", "exp", "cust", "other", "okind", "flip"]].drop_duplicates()
    custs = sorted(pt.cust.unique())
    ents  = sorted(pt[pt.okind == "CPTY"].other.unique())
    pr_list = [(str(c), str(o)) for c, o in pt[["cust", "other"]].drop_duplicates().itertuples(index=False)]
    st.step(f"{best}: {len(ms)} months · {pt.lid.nunique():,} TDH links · {len(pr_list):,} customer ↔ counterparty pairs · {len(ents):,} entities")

    def _rails():
        cmap = (spark.table(T_MAP).select(F.col("cpty_id").cast("string").alias("oid"), F.col("ent_l2").cast("string").alias("other"))
                     .join(F.broadcast(ids_df(ents, "other")), "other"))
        e = spark.table(T_EDGE).where(F.col("month").isin(ms))
        if "is_self" in HAVE[T_EDGE]:
            e = e.where(~F.coalesce(F.col("is_self"), F.lit(False)))
        sel = lambda c, o, k, d: e.where(F.col(c.replace("_id", "_kind")) == "CUST").select(
            F.col(c).cast("string").alias("cust"), F.col(o).cast("string").alias("oid"), F.col(k).alias("okind"), F.lit(d).alias("dir"),
            "category", F.col("amount").cast("double").alias("amount"), F.col("n_txn").cast("long").alias("n_txn"))
        u = (sel("src_id", "dst_id", "dst_kind", "out").unionByName(sel("dst_id", "src_id", "src_kind", "in"))
             .join(F.broadcast(ids_df(custs, "cust")), "cust"))
        keep = ["cust", "other", "okind", "dir", "category", "amount", "n_txn"]
        uc = u.where(F.col("okind") == "CUST").withColumn("other", F.col("oid")).select(*keep)
        ue = u.where(F.col("okind") == "CPTY").join(F.broadcast(cmap), "oid").select(*keep)
        c1 = F.substring(F.col("category").cast("string"), 1, 1)
        rail = (F.when(c1.isin("1", "2", "3"), "ACH").when(c1 == "4", "WIRE").when(c1.isin("5", "6"), "RTP")
                 .when(c1 == "7", "CHECK").when(c1 == "9", "PCARD").otherwise("OTHER"))
        prs = spark.createDataFrame(pr_list, "cust string, other string")
        return (uc.unionByName(ue).join(F.broadcast(prs), ["cust", "other"]).withColumn("rail", rail)
                  .groupBy("cust", "other", "okind", "dir", "rail").agg(F.sum("amount").alias("amount"), F.sum("n_txn").alias("n_txn")).toPandas())
    RL = cached("tdh_rails", [T_EDGE, T_MAP, ms, pr_list], _rails, st)

    x = RL.merge(pt, on=["cust", "other", "okind"])
    x["dir_dc"] = np.where(x.flip, x.dir.map({"in": "out", "out": "in"}), x.dir)        # to the data center's view
    x = x[x.dir_dc == x.exp]
    PV = (x.pivot_table(index="lid", columns="rail", values="amount", aggfunc="sum").reindex(index=Tf.lid, columns=RAILS).fillna(0.0)
          if len(x) else pd.DataFrame(0.0, index=Tf.lid, columns=RAILS))
    PV["ALL"] = PV[RAILS].sum(axis=1)
    tdh = Tf.wb_amount.to_numpy(dtype=float)
    def fit(name, pk):
        pk = pk.to_numpy(dtype=float); has = pk > 0
        r = np.where(has, pk / tdh, np.nan)
        return {"PKG dollars counted": name, "links with dollars": int(has.sum()), "PKG dollars": pk.sum(),
                "PKG / TDH total": pk.sum() / tdh.sum() if tdh.sum() else np.nan,
                "exact (±1%) %": float(np.nansum(np.abs(r - 1) <= 0.01) / len(tdh)), "within ±10% %": float(np.nansum(np.abs(r - 1) <= 0.10) / len(tdh))}
    r1t = pd.DataFrame([fit("ACH only", PV.ACH), fit("ACH + RTP", PV.ACH + PV.RTP), fit("all rails", PV.ALL),
                        fit("WIRE only", PV.WIRE), fit("CHECK only", PV.CHECK), fit("RTP only", PV.RTP), fit("PCARD only", PV.PCARD)])
    mix = pd.DataFrame([{"rail": r, "TDH links using it": int((PV[r] > 0).sum()), "PKG dollars": PV[r].sum(),
                         "share %": PV[r].sum() / PV.ALL.sum() if PV.ALL.sum() else np.nan} for r in RAILS if PV[r].sum() > 0])
    ex_all = (np.abs(PV.ALL.to_numpy() / tdh - 1) <= 0.01)
    ex_ach = (np.abs(PV.ACH.to_numpy() / tdh - 1) <= 0.01)
    verdict = pd.DataFrame([
        {"item": "found TDH links compared", "value": f"{len(Tf):,}"},
        {"item": "exact with all rails", "value": f"{int(ex_all.sum()):,}"},
        {"item": "exact with ACH only", "value": f"{int(ex_ach.sum()):,}"},
        {"item": "exact with ACH only but NOT with all rails (we hold more than TDH)", "value": f"{int((ex_ach & ~ex_all).sum()):,}"},
        {"item": "exact with all rails but NOT with ACH only (TDH holds non-ACH too)", "value": f"{int((ex_all & ~ex_ach).sum()):,}"},
        {"item": "PKG non-ACH dollars on these links (not in TDH if TDH is ACH only)", "value": f"{(PV.ALL - PV.ACH).sum():,.0f}"},
        {"item": "edge-table dollars / pair-table dollars for the same links and period (should be about 1)",
         "value": f"{PV.ALL.sum() / Tf.pkg_best.sum():.3f}" if Tf.pkg_best.sum() else "n/a"},
    ])
    TR = Tf[["lid", "master_name", "dc_name", "so_id", "so_name", "type", "wb_amount", "m_basis"]].merge(PV.reset_index(), on="lid")
    TR["ACH / TDH"] = TR.ACH / TR.wb_amount
    TR["ALL / TDH"] = TR.ALL / TR.wb_amount
    TR.drop(columns="lid").to_csv(Path(OUT_DIR) / "dc_tdh_rails.csv", index=False)
    more = (TR[TR.ALL > TR.ACH * 1.01].sort_values("wb_amount", ascending=False)
              [["master_name", "so_name", "type", "wb_amount", "ACH", "WIRE", "RTP", "CHECK", "ALL", "ACH / TDH", "ALL / TDH"]].head(TOP_N)
              .rename(columns={"master_name": "data center (parent)", "so_name": "second-order party", "wb_amount": "TDH amount", "ALL": "all rails"}))
    st.done((f"8a. Which rails reproduce the TDH amounts ({best}; shares are of the {len(Tf):,} found TDH links)", r1t),
            ("8b. Verdict counts", verdict), ("8c. PKG dollars on the found TDH links, by rail", mix),
            (f"8d. Links where PKG holds more than ACH (largest {TOP_N} by TDH amount)", more),
            ("Saved", f"{OUT_DIR}/dc_tdh_rails.csv: every found TDH link with PKG dollars per rail"))

## 9. Summary
The headline numbers of sections A to C in one table.

In [ ]:
# ── 9. Summary ──────────────────────────────────────────────────────────────
def sh(a, b):
    return f"{a:,} of {b:,} ({a / b:.1%})" if b else "0 of 0"
allL, tst = len(L), int(L.testable.sum())
S = [
    ("A. Workbook Power IDs found in pkg_cust_dim", sh(int(PID.found.sum()), len(PID))),
    ("A. … with payment metrics in the last 12 months", sh(int((PID.mdm_with_metrics > 0).sum()), len(PID))),
    ("A. … with a credit row (" + TM_LAST + ")", sh(int((PID.credit_source != "").sum()), len(PID))),
    ("A. … of which NOT in pkg_cust_dim but with a credit row", f"{int(((~PID.found) & (PID.credit_source != '')).sum()):,}"),
    ("A. … with DHE > 0", sh(int((PID.dhe.fillna(0) > 0).sum()), len(PID))),
    ("B. Names without a Power ID seen as PKG counterparties (by name)", sh(int(NM.drop_duplicates('id').matched.sum()), NM.id.nunique())),
    ("C. Workbook links that can be tested in PKG", sh(tst, allL)),
    ("C. Testable links with payments found", sh(int(L.found.sum()), tst)),
    ("C. … matched by Power IDs only (observed)", f"{int((L.found & (L.m_basis == 'ids')).sum()):,}"),
    ("C. … matched by name, fully or partly (inferred)", f"{int((L.found & (L.m_basis != 'ids')).sum()):,}"),
    (f"C. {TDH_TAB} links found", sh(int(T.found.sum()), len(T))),
    (f"C. {TDH_TAB} dollars on found links", f"{T.loc[T.found, 'wb_amount'].sum():,.0f} of {tdh_tot:,.0f}" + (f" ({T.loc[T.found, 'wb_amount'].sum() / tdh_tot:.1%})" if tdh_tot else "")),
    (f"C. Best-fitting PKG period for {TDH_TAB} amounts", str(best) if best else "n/a"),
]
if TR is not None:
    S += [(f"C. Found {TDH_TAB} links exact (±1%) with ACH only / with all rails", f"{int(ex_ach.sum()):,} / {int(ex_all.sum()):,} of {len(Tf):,}"),
          (f"C. PKG non-ACH dollars on found {TDH_TAB} links", f"{(PV.ALL - PV.ACH).sum():,.0f}")]
S.append((f"C. {TDH_TAB} links NOT found (listed in D5, saved in dc_tdh_not_found.csv)", f"{len(NF):,} links, {NF.wb_amount.sum():,.0f} dollars"))
show(("9. Summary", pd.DataFrame(S, columns=["check", "result"])),
     ("Files in " + OUT_DIR + "/", " · ".join(sorted(p.name for p in Path(OUT_DIR).glob("dc_*.csv")))))

## 10. Figures of the findings
Four figures saved as PNG files under `dc_check_out/figures/` (nothing is drawn inline; the numbers behind each figure are shown as tables).

In [ ]:
# ── 10. Figures of the findings ─────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")                                  # figures go to disk only
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter, FuncFormatter

FIG_DIR = Path(OUT_DIR) / "figures"
FIG_DIR.mkdir(exist_ok=True)
SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#b5b4ab"
RAMP = ["#184f95", "#3987e5", "#86b6ef"]               # one hue, dark → light (degrees of the same thing)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "text.color": INK, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK, "axes.edgecolor": GRID, "axes.grid": False})
FIGS = []

def new_fig(w, h, title, sub, top=0.80, n=1):
    fig, ax = plt.subplots(1, n, figsize=(w, h), dpi=160)
    fig.patch.set_facecolor(SURF)
    for a in (ax if n > 1 else [ax]):
        a.set_facecolor(SURF)
        for sp in ("top", "right"):
            a.spines[sp].set_visible(False)
        a.tick_params(length=0)
    fig.text(0.02, 0.975, title, fontsize=12.5, fontweight="bold", color=INK, va="top")
    fig.text(0.02, 0.975 - 0.42 / h, sub, fontsize=9, color=INK2, va="top")
    fig.subplots_adjust(top=top, left=0.02, right=0.98)
    return fig, ax

def save_fig(fig, name, what):
    path = FIG_DIR / name
    fig.savefig(path, facecolor=SURF, bbox_inches="tight", pad_inches=0.25)
    plt.close(fig)
    FIGS.append({"file": str(path), "shows": what})

def _dark(c):
    r, g, b = (int(c[i:i + 2], 16) / 255 for i in (1, 3, 5))
    return 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.55

def seg_bars(ax, rows, parts, colors):
    """100% bars split into parts; the share is printed on a part when it fits; names go in the legend."""
    for i, (lab, vals) in enumerate(rows):
        tot, left = float(sum(vals)), 0.0
        for v, c in zip(vals, colors):
            sh = v / tot if tot else 0.0
            if sh > 0:
                ax.barh(i, sh, left=left, color=c, edgecolor=SURF, linewidth=2, height=0.62)
            if sh >= 0.06:
                ax.text(left + sh / 2, i, f"{sh:.0%}", ha="center", va="center", fontsize=8.5, color="white" if _dark(c) else INK)
            left += sh
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.spines["left"].set_visible(False)
    ax.legend(handles=[Patch(facecolor=c, label=p) for p, c in zip(parts, colors)], loc="upper left", bbox_to_anchor=(0, -0.10),
              ncol=min(len(parts), 3), frameon=False, fontsize=8.5, handlelength=1.2, columnspacing=1.6)

def money(x, _=None):
    return f"{x / 1e9:.2f}B" if abs(x) >= 1e9 else f"{x / 1e6:.0f}M" if abs(x) >= 1e6 else f"{x / 1e3:.0f}K" if abs(x) >= 1e3 else f"{x:.0f}"

blocks = []
# ---- figure 1: how much of TDH reconciles with our payments --------------------------------------------------------
period = best if best else "no period"
if len(T):
    ach = (PV.ACH.reindex(Tf.lid).to_numpy(dtype=float) if TR is not None else Tf.pkg_best.to_numpy(dtype=float)) if len(Tf) else np.array([])
    basis = "ACH" if TR is not None else "all rails"
    r = ach / Tf.wb_amount.to_numpy(dtype=float) if len(Tf) else np.array([])
    cat = np.select([np.abs(r - 1) <= 0.01, np.abs(r - 1) <= 0.10, r == 0], ["a", "b", "d"], default="c")
    P1 = ["same amount (within 1%)", "close (within 10%)", "found, amount differs", "found, but no dollars in that period", "not found"]
    C1 = RAMP + [ORANGE, GRAY]
    n_ = [int((cat == k).sum()) for k in "abcd"] + [int((~T.found).sum())]
    d_ = [float(Tf.wb_amount[cat == k].sum()) for k in "abcd"] + [float(T.loc[~T.found, "wb_amount"].sum())]
    fig, ax = new_fig(8.6, 3.3, "TDH flows against our own payment data",
                      f"{len(T):,} TDH links, {money(T.wb_amount.sum())} dollars · our {basis} dollars, {period}", top=0.74)
    seg_bars(ax, [(f"links ({len(T):,})", n_), (f"dollars ({money(T.wb_amount.sum())})", d_)], P1, C1)
    save_fig(fig, "fig1_tdh_reconciliation.png", "share of TDH links and dollars that reconcile with PKG")
    blocks.append(("Figure 1 numbers", pd.DataFrame({"TDH links": P1, "links": n_, "TDH dollars": d_})))

    # ---- figure 2: TDH amount against our amount, link by link ------------------------------------------------------
    ok = ach > 0
    if ok.any():
        fig, ax = new_fig(6.4, 6.2, "Link by link: TDH amount against our dollars",
                          f"{int(ok.sum()):,} found TDH links with dollars in {period} · points on the line are equal amounts", top=0.86)
        ids_ = (Tf.m_basis == "ids").to_numpy()
        for mask, c, lab in ((ok & ~ids_, ORANGE, "one side matched by name"), (ok & ids_, BLUE, "both sides matched by Power ID")):
            ax.scatter(Tf.wb_amount.to_numpy(dtype=float)[mask], ach[mask], s=30, color=c, alpha=0.75, edgecolor=SURF, linewidth=0.8,
                       label=f"{lab} ({int(mask.sum()):,})", zorder=3)
        lo = max(min(Tf.wb_amount[ok].min(), ach[ok].min()) / 2, 0.5)
        hi = max(Tf.wb_amount[ok].max(), ach[ok].max()) * 2
        ax.plot([lo, hi], [lo, hi], color=INK2, linewidth=1, zorder=2)
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.xaxis.set_major_formatter(FuncFormatter(money)); ax.yaxis.set_major_formatter(FuncFormatter(money))
        ax.grid(True, color=GRID, linewidth=0.8, zorder=0); ax.set_axisbelow(True)
        ax.set_xlabel("TDH amount (workbook)"); ax.set_ylabel(f"our {basis} dollars, {period}")
        ax.legend(loc="upper left", frameon=False, fontsize=8.5)
        fig.subplots_adjust(left=0.13, bottom=0.10)
        save_fig(fig, "fig2_tdh_vs_pkg_scatter.png", "each found TDH link: workbook amount against PKG dollars")

# ---- figure 3: every workbook link, by tab -------------------------------------------------------------------------
G3 = {"5": 0, "6": 0, "4": 1, "2": 2, "3": 2, "0": 3, "1": 3}
P3 = ["payments found in PKG", "both sides seen, no payments between them", "one side not seen in PKG", "PKG cannot see it (neither side banks at PNC, or no named data center)"]
g3 = L.assign(g=L.status.str[0].map(G3)).groupby(["tab", "g"]).size().unstack(fill_value=0).reindex(index=TABS, columns=range(4), fill_value=0)
fig, ax = new_fig(8.6, 0.5 * len(TABS) + 2.2, "Which workbook links show up as payments",
                  f"{len(L):,} links (data center ↔ second-order party) · payments {PAIR_MONTH_LIST[0]} to {PAIR_MONTH_LIST[-1]}", top=0.80)
seg_bars(ax, [(f"{t} ({int(g3.loc[t].sum()):,})", g3.loc[t].tolist()) for t in TABS], P3, [BLUE, ORANGE, AQUA, GRAY])
ax.legend(handles=[Patch(facecolor=c, label=p_) for p_, c in zip(P3, [BLUE, ORANGE, AQUA, GRAY])], loc="upper left",
          bbox_to_anchor=(0, -0.12), ncol=1, frameon=False, fontsize=8.5, handlelength=1.2)
save_fig(fig, "fig3_links_by_tab.png", "status of every workbook link, by tab")
t3 = g3.copy(); t3.columns = ["payments found", "both seen, no payments", "one side not seen", "PKG cannot see it"]
blocks.append(("Figure 3 numbers", t3.rename_axis("tab").reset_index()))

# ---- figure 4: where the unreconciled TDH dollars sit ---------------------------------------------------------------
if len(NF):
    by = NF.groupby(NF.master_name.fillna(NF.dc_name)).wb_amount.sum().sort_values(ascending=False)
    top4 = by.head(10)
    rest = by.iloc[10:].sum()
    labs = list(top4.index) + ([f"all other data centers ({len(by) - 10})"] if len(by) > 10 else [])
    vals = list(top4.values) + ([rest] if len(by) > 10 else [])
    fig, ax = new_fig(7.6, 0.36 * len(labs) + 1.6, "TDH dollars we have not found yet, by data center",
                      f"{len(NF):,} links, {money(NF.wb_amount.sum())} dollars · top 5 hold "
                      f"{by.head(5).sum() / by.sum():.0%}", top=1 - 0.95 / (0.36 * len(labs) + 1.6))
    ax.barh(range(len(labs)), vals, color=[BLUE] * len(top4) + [GRAY] * (len(labs) - len(top4)), height=0.62)
    for i, v in enumerate(vals):
        ax.text(v, i, "  " + money(v), va="center", ha="left", fontsize=8.5, color=INK)
    ax.set_yticks(range(len(labs))); ax.set_yticklabels([str(x)[:38] for x in labs]); ax.invert_yaxis()
    ax.set_xlim(0, max(vals) * 1.15); ax.xaxis.set_visible(False)
    for sp in ("bottom", "left"):
        ax.spines[sp].set_visible(False)
    fig.subplots_adjust(left=0.30)
    save_fig(fig, "fig4_tdh_not_found_by_dc.png", "TDH dollars not found, by data center")
show(("10. Figures saved (open them from the file browser)", pd.DataFrame(FIGS)), *blocks)

## 11. Are the payments on the found links regular or casual?
For every found link, the customer ↔ counterparty pairs behind it are listed (saved as `dc_found_pairs.csv`, which the app will need). A **stream** is one pair in one direction.
Two views of the same streams:
- **Stored**: our periodicity table `pkg_cust_reg_stream`, which already holds one row per customer × direction × counterparty × month for streams that are regular (cadence, late, overdue, stopped, next expected date). A stream that is not in it is not regular.
- **On the fly**: a short rule applied in pandas to the payment days from `pkg_edge_daily`, the way an app would do it without Spark. It is timed, and compared with the stored result.

In [ ]:
# ── 11. Regularity of the payments on the found links ───────────────────────
st = Status("Regularity of the found pairs")
STATE3 = ["on schedule", "late or overdue", "stopped"]
CADS   = ["daily", "weekly", "biweekly", "monthly", "quarterly"]

# ---- the pairs behind every found link ------------------------------------------------------------------------------
PCc = ["cust", "other", "so_id", "okind", "flip", "route"]
q1 = INT_M[["cust", "other"]].drop_duplicates().merge(so_map, on="other").assign(okind="CUST", flip=False, route="1 both bank at PNC")[PCc]
q2 = (EXT_M[["cust", "other"]].drop_duplicates().merge(ent2key, on="other").merge(so_nm, on="key")
      .assign(okind="CPTY", flip=False, route="2 party matched by name")[PCc].drop_duplicates())
q3 = PM[["cust", "other", "so_id", "key"]].drop_duplicates().assign(okind="CPTY", flip=True, route="3 data center matched by name")
def link_pairs(dcmap, dcnames):
    a = pd.concat([q1, q2], ignore_index=True).merge(dcmap, on="cust")
    b = q3.merge(dcnames, on="key")
    return pd.concat([a, b], ignore_index=True)[["dc_key"] + PCc].drop_duplicates()
FLK = L[L.found].reset_index(drop=True)
FLK["link"] = FLK.index
lc = ["link", "tab", "type", "master_id", "master_name", "dc_id", "dc_name", "so_id", "so_name"]
LP = pd.concat([FLK[FLK.status.str[0] == "5"][lc].merge(link_pairs(dc_e, dcn_e), left_on=["dc_id", "so_id"], right_on=["dc_key", "so_id"]),
                FLK[FLK.status.str[0] == "6"][lc].merge(link_pairs(dc_p, dcn_p), left_on=["master_id", "so_id"], right_on=["dc_key", "so_id"])],
               ignore_index=True).drop(columns="dc_key")
LP.to_csv(Path(OUT_DIR) / "dc_found_pairs.csv", index=False)
# one row per pair, described from the data center's side
PR = (LP.assign(dc=LP.master_name.fillna(LP.dc_name))
        .groupby(["cust", "other", "okind"], sort=False)
        .agg(flip=("flip", "first"), route=("route", "first"), dc=("dc", "first"), party=("so_name", "first"),
             type=("type", lambda x: " + ".join(sorted(set(x)))), tabs=("tab", lambda x: " + ".join(sorted(set(x)))),
             links=("link", "nunique")).reset_index())
custs   = sorted(PR.cust.unique())
ents    = sorted(PR[PR.okind == "CPTY"].other.unique())
pr_list = [(str(c), str(o)) for c, o in PR[["cust", "other"]].drop_duplicates().itertuples(index=False)]
st.step(f"{len(FLK):,} found links → {len(PR):,} customer ↔ counterparty pairs")

# ---- payment days of those pairs (Spark, once) ----------------------------------------------------------------------
D = None
if OK.get(T_EDGE_D) and OK.get(T_MAP) and pr_list:
    dms = MONTHS[T_EDGE_D][-DAILY_MONTHS:]
    def _daily():
        cmap = (spark.table(T_MAP).select(F.col("cpty_id").cast("string").alias("oid"), F.col("ent_l2").cast("string").alias("other"))
                     .join(F.broadcast(ids_df(ents, "other")), "other"))
        e = spark.table(T_EDGE_D).where(F.col("month").isin(dms))
        if "is_self" in HAVE[T_EDGE_D]:
            e = e.where(~F.coalesce(F.col("is_self"), F.lit(False)))
        sel = lambda c, o, k, d: e.where(F.col(c.replace("_id", "_kind")) == "CUST").select(
            F.col(c).cast("string").alias("cust"), F.col(o).cast("string").alias("oid"), F.col(k).alias("okind"), F.lit(d).alias("dir"),
            F.col("txn_date").cast("string").alias("txn_date"), F.col("amount").cast("double").alias("amount"), F.col("n_txn").cast("long").alias("n_txn"))
        u = (sel("src_id", "dst_id", "dst_kind", "out").unionByName(sel("dst_id", "src_id", "src_kind", "in"))
             .join(F.broadcast(ids_df(custs, "cust")), "cust"))
        keep = ["cust", "other", "okind", "dir", "txn_date", "amount", "n_txn"]
        uc = u.where(F.col("okind") == "CUST").withColumn("other", F.col("oid")).select(*keep)
        ue = u.where(F.col("okind") == "CPTY").join(F.broadcast(cmap), "oid").select(*keep)
        prs = spark.createDataFrame(pr_list, "cust string, other string")
        return (uc.unionByName(ue).join(F.broadcast(prs), ["cust", "other"])
                  .groupBy("cust", "other", "okind", "dir", "txn_date").agg(F.sum("amount").alias("amount"), F.sum("n_txn").alias("n_txn")).toPandas())
    D = cached("pair_days", [T_EDGE_D, T_MAP, dms, pr_list], _daily, st)
    D["txn_date"] = pd.to_datetime(D.txn_date)
    _y, _m = int(dms[-1][:4]), int(dms[-1][5:7])
    END = pd.Timestamp(_y, _m, 1) + pd.offsets.MonthEnd(0)                 # last day covered by the payment data
    for c in ("cust", "other", "okind", "dir"):
        D[c] = D[c].astype(object)

# ---- stored periodicity for the same pairs (Spark, once) ------------------------------------------------------------
REG = None
if OK.get(T_REG) and pr_list:
    rms = MONTHS[T_REG][-REG_MONTHS:]
    ropt = HAVE[T_REG]
    def _reg():
        prs = spark.createDataFrame(pr_list, "cust string, other string")
        return (spark.table(T_REG).where(F.col("month").isin(rms))
                     .select(F.col("cust").cast("string").alias("cust"), "dir", F.col("other").cast("string").alias("other"), "month",
                             F.col("cadence").cast("string").alias("cadence"), *[F.col(c).cast("string").alias(c) for c in ropt])
                     .join(F.broadcast(prs), ["cust", "other"]).toPandas())
    REG = cached("pair_reg", [T_REG, rms, pr_list, ropt], _reg, st)
    for c in ("cust", "other", "dir", "cadence"):
        REG[c] = REG[c].astype(object)

# ---- on-the-fly rule (pandas only): the same idea as the periodicity build, kept short ------------------------------
# regular = at least 4 payment days in the window, median gap on a known cadence, at least 70% of gaps on time
FLY_CAD = [("daily", 1, 3, 1, 4), ("weekly", 5, 9, 7, 3), ("biweekly", 12, 17, 14, 4), ("monthly", 26, 35, 30.4, 6), ("quarterly", 80, 100, 91, 15)]
def fly_rule(days, end):
    """days: sorted unique payment days (numpy datetime64[D]). Returns cadence, state and the numbers behind them."""
    n, last = len(days), days[-1]
    since = int((end - last) / np.timedelta64(1, "D"))
    out = {"n_days": n, "first_dt": days[0], "last_dt": last, "days_since": since, "fly_cadence": "casual", "fly_state": "casual",
           "med_gap": np.nan, "ontime": np.nan}
    if n >= 4:
        gaps = np.diff(days).astype("timedelta64[D]").astype(int)
        med = float(np.median(gaps))
        out["med_gap"] = med
        for name, lo, hi, g, tol in FLY_CAD:
            if lo <= med <= hi:
                on = float(np.mean(gaps <= g + tol) if name == "daily" else np.mean(np.abs(gaps - g) <= tol))
                out["ontime"] = on
                if on >= 0.70:
                    out["fly_cadence"] = name
                    out["fly_state"] = ("on schedule" if since <= g + tol else
                                        "late or overdue" if since <= max(3 * g, g + 30) else "stopped")
                break
    return out

S, fly_secs = None, np.nan
if D is not None and len(D):
    endd = np.datetime64(END.date())
    t0 = time.perf_counter()
    rows_ = []
    for (c, o, k, d), g in D.groupby(["cust", "other", "okind", "dir"], sort=False):
        days = np.unique(g.txn_date.values.astype("datetime64[D]"))
        rows_.append({"cust": c, "other": o, "okind": k, "dir": d, **fly_rule(days, endd)})
    S = pd.DataFrame(rows_)
    fly_secs = time.perf_counter() - t0
    a12 = (D[D.txn_date > END - pd.DateOffset(months=12)].groupby(["cust", "other", "okind", "dir"]).amount.sum()
             .rename("amt_12m").reset_index())
    S = S.merge(a12, on=["cust", "other", "okind", "dir"], how="left").fillna({"amt_12m": 0.0})
elif REG is not None and len(REG):
    S = REG[["cust", "other", "dir"]].drop_duplicates().merge(PR[["cust", "other", "okind"]], on=["cust", "other"])
    S["amt_12m"] = np.nan

blocks = []
if S is None or not len(S):
    st.done(("11. Regularity", "Skipped: neither pkg_edge_daily nor pkg_cust_reg_stream gave rows for the found pairs (see pre-flight)."))
else:
    S = S.merge(PR, on=["cust", "other", "okind"], how="left")
    to_dc = np.where(S.flip, S.dir == "out", S.dir == "in")                 # is the data center the one being paid?
    S["direction"] = np.where(to_dc, "paid to the data center", "paid by the data center")
    _dc = S.dc.astype(str) + np.where(S.route.str[0] == "3", " *", "")        # * = account at another bank, matched by name
    _pt = S.party.astype(str) + np.where(S.route.str[0] == "2", " *", "")
    S["payer"], S["payee"] = np.where(to_dc, _pt, _dc), np.where(to_dc, _dc, _pt)

    # stored view: the latest row of each stream in the periodicity table
    if REG is not None and len(REG):
        yes = lambda col: (REG[col].astype(str).str.lower().isin(["1", "1.0", "true", "y", "yes"]) if col in REG else pd.Series(False, index=REG.index))
        REG["_stop"], REG["_over"] = yes("stopped"), yes("overdue")
        REG["_late"] = yes("late") | (yes("skipped") & (REG.cadence != "daily"))     # "skipped" is not meaningful for daily streams (v1.1)
        lastr = REG.sort_values("month").groupby(["cust", "dir", "other"]).tail(1)
        _st = (lastr.status.astype(str).str.strip().str.lower() if "status" in lastr else pd.Series("", index=lastr.index))
        LAPSED = "no longer regular"
        lastr = lastr.assign(reg_state=np.select(
            [_st == "not regular", _st == "stopped", _st == "overdue", lastr._stop, lastr.month < rms[-1], lastr._over | lastr._late],
            [LAPSED, "stopped", "late or overdue", "stopped", LAPSED, "late or overdue"], default="on schedule"))
        # a stream whose latest row says "not regular", or that left the table before the latest month without a stop, is not regular now
        keepc = ["cust", "dir", "other", "cadence", "reg_state", "month"] + [c for c in ("status", "next_dt", "last_dt") if c in lastr]
        S = S.merge(lastr[keepc].rename(columns={"cadence": "reg_cadence", "month": "reg_last_month", "last_dt": "reg_last_dt"}),
                    on=["cust", "dir", "other"], how="left")
        S["reg_cadence_was"] = S.reg_cadence
        _lap = S.reg_state == LAPSED
        S["reg_cadence"] = S.reg_cadence.mask(_lap).fillna("not regular")
        S["reg_state"] = S.reg_state.fillna("not regular")
    USE = "stored" if "reg_cadence" in S else "fly"
    S["cadence"] = S.reg_cadence.replace("not regular", "casual") if USE == "stored" else S.fly_cadence
    S["state"] = S.reg_state.replace({"not regular": "casual", "no longer regular": "casual"}) if USE == "stored" else S.fly_state
    S.to_csv(Path(OUT_DIR) / "dc_stream_regularity.csv", index=False)

    def mix(cad, state, title):
        t = (S.groupby([cad, state]).agg(streams=("cust", "size"), usd=("amt_12m", "sum")).reset_index())
        w = t.pivot(index=cad, columns=state, values="streams").fillna(0).astype(int).rename_axis(None, axis=1)
        w = w.reindex(index=[c for c in CADS + ["casual", "not regular"] if c in w.index] + [c for c in w.index if c not in CADS + ["casual", "not regular"]])
        w = w[[c for c in STATE3 + ["casual", "no longer regular", "not regular"] if c in w.columns]]
        w["streams"] = w.sum(axis=1)
        w["dollars, last 12 months"] = t.groupby(cad).usd.sum().reindex(w.index)
        tot = w.sum().to_frame().T; tot.index = ["ALL"]
        return (title, pd.concat([w, tot]).rename_axis("cadence").reset_index())
    reg_n = int((S.cadence != "casual").sum())
    reg_usd = S.loc[S.cadence != "casual", "amt_12m"].sum()
    head = pd.DataFrame([
        {"item": "found links", "value": f"{len(FLK):,}"},
        {"item": "customer ↔ counterparty pairs behind them", "value": f"{len(PR):,}"},
        {"item": f"streams (pair × direction) with payments, {dms[0]} to {dms[-1]}" if D is not None else "streams", "value": f"{len(S):,}"},
        {"item": "regular streams", "value": f"{reg_n:,} ({reg_n / len(S):.1%})"},
        {"item": "dollars on regular streams, last 12 months", "value": (f"{reg_usd:,.0f} of {S.amt_12m.sum():,.0f} ({reg_usd / S.amt_12m.sum():.1%})"
                                                                         if S.amt_12m.sum() > 0 else "n/a")},
        {"item": "regularity taken from", "value": "pkg_cust_reg_stream (stored)" if USE == "stored" else "on-the-fly rule (stored table not available)"},
    ])
    blocks.append(("R1. Streams behind the found links", head))
    if USE == "stored":
        blocks.append((f"R2. Stored periodicity ({rms[0]} to {rms[-1]}): cadence by current state (number of streams)", mix("reg_cadence", "reg_state", "")[1]))
        if "status" in S:
            hv = S[S.reg_last_month.notna()]
            sv = (hv.assign(status=hv.status.fillna("(blank)")).groupby(["status", "reg_state"]).size().unstack(fill_value=0)
                    .rename_axis(None, axis=1).rename_axis("status as stored (latest row)").reset_index())
            blocks.append((f"R2b. Stored status of the {len(hv):,} streams that have a row, and the state used here", sv))
        if "days_since" in S:
            rg = S[S.reg_cadence != "not regular"]
            bk = pd.cut(rg.days_since, [-1, 7, 35, 100, 10**6], labels=["0 to 7 days", "8 to 35 days", "36 to 100 days", "over 100 days"])
            qa = (rg.groupby([rg.reg_state, bk], observed=False).size().unstack(fill_value=0).rename_axis(None, axis=1)
                    .rename_axis("state (stored)").reset_index())
            blocks.append((f"R2c. Check: days since the last payment (to {END.date()}), by stored state", qa))
    if "fly_cadence" in S:
        blocks.append((f"R3. On-the-fly rule, payment days {dms[0]} to {dms[-1]}: cadence by current state (number of streams)", mix("fly_cadence", "fly_state", "")[1]))
        cost = pd.DataFrame([
            {"item": "payment-day rows read for these pairs", "value": f"{len(D):,}"},
            {"item": "streams classified", "value": f"{len(S):,}"},
            {"item": "pandas time for the rule", "value": f"{fly_secs:.2f} seconds"},
            {"item": "payment-day rows per pair (median / largest)", "value": f"{int(D.groupby(['cust', 'other']).size().median()):,} / {int(D.groupby(['cust', 'other']).size().max()):,}"},
        ] + ([{"item": "stored rows read for these pairs", "value": f"{len(REG):,}"}] if REG is not None else []))
        blocks.append(("R4. Cost of doing it on the fly", cost))
    if USE == "stored" and "fly_cadence" in S:
        a, b = S.reg_cadence != "not regular", S.fly_cadence != "casual"
        agree = pd.DataFrame([
            {"stored": "regular", "on the fly": "regular", "streams": int((a & b).sum()), "dollars, last 12 months": S.amt_12m[a & b].sum()},
            {"stored": "regular", "on the fly": "casual", "streams": int((a & ~b).sum()), "dollars, last 12 months": S.amt_12m[a & ~b].sum()},
            {"stored": "not regular", "on the fly": "regular", "streams": int((~a & b).sum()), "dollars, last 12 months": S.amt_12m[~a & b].sum()},
            {"stored": "not regular", "on the fly": "casual", "streams": int((~a & ~b).sum()), "dollars, last 12 months": S.amt_12m[~a & ~b].sum()}])
        both = a & b
        note = (f"Same regular / casual call on {((a == b).mean()):.1%} of streams. Where both say regular: same cadence "
                f"{(S.reg_cadence[both] == S.fly_cadence[both]).mean():.1%}, same state {(S.reg_state[both] == S.fly_state[both]).mean():.1%}."
                if both.any() else f"Same regular / casual call on {((a == b).mean()):.1%} of streams.")
        blocks += [("R5. Stored against on the fly", agree), ("R5 note", note)]
    bt = (S.assign(reg=S.cadence != "casual").groupby(["type", "direction"])
            .agg(streams=("cust", "size"), regular=("reg", "sum"), usd=("amt_12m", "sum"),
                 usd_reg=("amt_12m", lambda x: x[S.loc[x.index, "cadence"] != "casual"].sum())).reset_index())
    bt["regular %"] = bt.regular / bt.streams
    bt["dollars regular %"] = np.where(bt.usd > 0, bt.usd_reg / bt.usd, np.nan)
    bt = bt.rename(columns={"type": "relationship type", "usd": "dollars, last 12 months"}).drop(columns="usd_reg").sort_values("streams", ascending=False).head(12)
    blocks.append(("R6. Regular share by relationship type and direction", bt))
    tcols = ["payer", "payee", "cadence", "state", "amt_12m"] + [c for c in ("n_days", "last_dt", "days_since", "next_dt") if c in S]
    topS = (S[S.cadence != "casual"].sort_values("amt_12m", ascending=False)[tcols].head(TOP_N)
              .rename(columns={"amt_12m": "dollars, last 12 months", "n_days": "payment days", "last_dt": "last payment",
                               "days_since": "days since", "next_dt": "next expected"}))
    if "last payment" in topS:
        topS["last payment"] = pd.to_datetime(topS["last payment"]).dt.strftime("%Y-%m-%d")
    blocks.append((f"R7. Largest regular streams (top {TOP_N}); * = account at another bank, matched by name", topS))
    blocks.append(("Saved", f"{OUT_DIR}/dc_found_pairs.csv (pairs behind every found link) · {OUT_DIR}/dc_stream_regularity.csv (one row per stream)"))
    st.done(*blocks)

## 12. Regularity figures
Two more PNG files: the mix of regular and casual streams with their current state, and the payment days of the largest streams, coloured the way the graph could colour its links.

In [ ]:
# ── 12. Regularity figures ──────────────────────────────────────────────────
SC = {"on schedule": "#0ca30c", "late or overdue": "#ec835a", "stopped": "#d03b3b", "casual": "#8f8e86"}
FIGS2 = []
if S is None or not len(S):
    show(("12. Regularity figures", "Skipped: no streams (see section 11)."))
else:
    src = "stored periodicity table" if USE == "stored" else "on-the-fly rule"
    # ---- figure 5: cadence by state, streams and dollars ----------------------------------------------------------
    order = [c for c in CADS if (S.cadence == c).any()] or ["casual"]
    states = [x for x in STATE3 + ["casual"] if (S.state[S.cadence.isin(order)] == x).any()]
    _cas = S[S.cadence == "casual"]
    fig, axs = new_fig(9.6, 0.42 * len(order) + 2.3, "Regular payments on the found links, and where they stand",
                       f"{int(S.cadence.isin(order).sum()):,} regular streams shown, from the {src} · not shown: {len(_cas):,} casual streams, "
                       f"{money(_cas.amt_12m.sum())} dollars", top=1 - 1.0 / (0.42 * len(order) + 2.3), n=2)
    for ax, col, lab in ((axs[0], None, "number of streams"), (axs[1], "amt_12m", "dollars, last 12 months")):
        w = (S.groupby(["cadence", "state"]).size() if col is None else S.groupby(["cadence", "state"])[col].sum()).unstack(fill_value=0)
        w = w.reindex(index=order, columns=states, fill_value=0)
        left = np.zeros(len(order))
        for x in states:
            ax.barh(range(len(order)), w[x].values, left=left, color=SC[x], edgecolor=SURF, linewidth=2, height=0.62)
            left += w[x].values
        for i, v in enumerate(left):
            ax.text(v, i, "  " + (f"{int(v):,}" if col is None else money(v)), va="center", ha="left", fontsize=8.5, color=INK)
        ax.set_yticks(range(len(order))); ax.set_yticklabels(order if ax is axs[0] else [""] * len(order)); ax.invert_yaxis()
        ax.set_xlim(0, max(left.max(), 1) * 1.18); ax.xaxis.set_visible(False); ax.set_title(lab, fontsize=9, color=INK2, loc="left")
        for sp in ("bottom", "left"):
            ax.spines[sp].set_visible(False)
    axs[0].legend(handles=[Patch(facecolor=SC[x], label=x) for x in states], loc="upper left", bbox_to_anchor=(0, -0.04), ncol=len(states),
                  frameon=False, fontsize=8.5, handlelength=1.2)
    fig.subplots_adjust(left=0.10, wspace=0.12)
    path = FIG_DIR / "fig5_regularity_mix.png"
    fig.savefig(path, facecolor=SURF, bbox_inches="tight", pad_inches=0.25); plt.close(fig)
    FIGS2.append({"file": str(path), "shows": "streams and dollars by cadence and current state"})

    # ---- figure 6: payment days of the largest streams ------------------------------------------------------------
    if D is not None and len(D):
        topT = S.sort_values("amt_12m", ascending=False).head(TIMELINE_N).reset_index(drop=True)
        h = 0.30 * len(topT) + 2.0
        fig, ax = new_fig(11.5, h, f"Payment days of the {len(topT)} largest streams",
                          f"one tick per payment day, {dms[0]} to {dms[-1]} · colour = current state ({src}) · * = account at another bank, matched by name",
                          top=1 - 0.95 / h)
        for i, r_ in topT.iterrows():
            dd = D[(D.cust == r_.cust) & (D.other == r_.other) & (D.dir == r_.dir)].txn_date.values
            ax.scatter(dd, np.full(len(dd), i), marker="|", s=70, linewidths=1.5, color=SC.get(r_.state, SC["casual"]), zorder=3)
            ax.text(END + pd.Timedelta(days=12), i, f"{r_.cadence} · {r_.state}" if r_.cadence != "casual" else "casual",
                    va="center", ha="left", fontsize=8, color=INK2)
        ax.set_yticks(range(len(topT)))
        ax.set_yticklabels([f"{str(a)[:28]} → {str(b)[:28]}" for a, b in zip(topT.payer, topT.payee)], fontsize=8)
        ax.invert_yaxis(); ax.set_ylim(len(topT) - 0.4, -0.6)
        ax.axvline(END, color=INK2, linewidth=1, zorder=2)
        ax.text(END, -0.75, "data ends", ha="center", va="bottom", fontsize=8, color=INK2)
        ax.set_xlim(pd.Timestamp(dms[0] + "-01") - pd.Timedelta(days=10), END + pd.Timedelta(days=150))
        ax.grid(True, axis="y", color=GRID, linewidth=0.6, zorder=0); ax.set_axisbelow(True)
        ax.spines["left"].set_visible(False)
        ax.legend(handles=[Patch(facecolor=SC[x], label=x) for x in STATE3 + ["casual"] if (topT.state == x).any()], loc="upper left",
                  bbox_to_anchor=(0, -0.06), ncol=4, frameon=False, fontsize=8.5, handlelength=1.2)
        fig.subplots_adjust(left=0.30)
        path = FIG_DIR / "fig6_payment_timelines.png"
        fig.savefig(path, facecolor=SURF, bbox_inches="tight", pad_inches=0.25); plt.close(fig)
        FIGS2.append({"file": str(path), "shows": "payment days of the largest streams, coloured by state"})
    show(("12. Regularity figures saved", pd.DataFrame(FIGS2)))